# Chess — Bayesian-MCTS ThompsonZero Training

Ports the `claude/bayesian_mcts_3` Boop notebook's algorithm — Thompson
sampling over per-action Dirichlet value beliefs, mixture-propagated (not
counted, not maxed) up the tree, trained with a non-self-referential evidence
loss — to **chess**, using OpenSpiel's native C++ chess implementation
(`games/chess/`) directly. No custom game engine is written here, unlike the
Boop notebooks: chess is already fast (sub-microsecond `clone()`), and its
`is_terminal()`/`returns()` already detect every real draw (insufficient
material, threefold repetition, stalemate, the 50-move rule) exactly, with no
move-cap heuristic needed.

### The one real generalization: Beta → Dirichlet (chess has draws)

Boop had no draws, so its network predicted (win-probability, confidence) and
every belief was a **Beta** over `P(win)`. Chess does, so this notebook's
network predicts **(win-probability, draw-probability, confidence)** and
every belief is a **Dirichlet** over (win, draw, loss) — recovering the
3-outcome design from the very first ThompsonZero notebook, but now built on
`bayesian_mcts_3`'s machinery (mixture propagation + evidence loss) instead of
naive count-accumulation.

The subtle part is *how* a 3-outcome Dirichlet belief is represented and
propagated through the tree without needing a 2D simplex grid (which would
lose all of the 1D CDF-product tricks mixture propagation depends on). This
notebook tracks, per edge:
- a **value** belief `v = p_win − p_loss ∈ [−1, 1]` on the same G-point grid
  machinery as Boop (just reinterpreted over `(−1, 1)` instead of `(0, 1)` via
  an affine map — `v` is exactly chess's own utility scale, `WinUtility=+1,
  DrawUtility=0, LossUtility=−1`, so no rescaling is needed anywhere);
- a **scalar mean draw-probability**, mixture-propagated by the *same*
  P(best) weights as the value (whichever child is likely to be chosen
  determines both its value and its draw-rate), with **no sign flip** (a draw
  stays a draw regardless of whose move it is — only `v` flips going up a ply).

Constructing a leaf's `v`-belief from the network's `(p_win, p_draw, conf)`
needs care: `v = (1 − d)·(2q − 1)` where `d = p_draw ~ Beta(α_d, α_w+α_l)` and
`q = p_win/(p_win+p_loss) ~ Beta(α_w, α_l)` are **independent** exact Dirichlet
marginals (the standard stick-breaking/aggregation property). `E[v]` and
`Var[v]` are computed in closed form from that decomposition and moment-matched
to a Beta — i.e. the `(1−d)` shrinkage is baked in at construction (a
likely-draw position is correctly represented as having a *narrower*, less
decisive value spread, not just a lower mean). Selection/argmax only ever
needs the scalar `v` (draws contribute exactly 0 to expected utility either
way), so nothing relevant to search is lost; only the training target needs
the full 3-way reconstruction, done once at the root by inverting the same
relationship. **Both directions were verified numerically** — against 2–4M
sample Monte-Carlo draws from actual Dirichlets across 8 parameter regimes
(uniform to extreme skew): closed-form `E[v]`/`Var[v]` matched to <1e-4, and
the full construct→reconstruct round trip recovered the original
`(p_win, p_draw, p_loss)` to <2e-3. MCTS-Solver was also extended to a genuine
3-way precedence (`WIN > DRAW > LOSS` for the mover; a proven draw beats a
proven loss) and end-to-end tested against real forced-mate and forced-draw
chess positions.

### What's identical to `bayesian_mcts_3`
- **Mixture propagation**: a node's value is the P(best)-weighted mixture over
  its children (not the hard max — that over-estimates under uncertainty, the
  maximization bias / optimizer's curse), recomputed from the current subtree
  on every backup. Nothing is counted or sampled as a "rollout."
- **Evidence loss**: training minimizes the negative Dirichlet-Multinomial
  marginal log-likelihood of the search evidence under the predicted prior —
  scored *against* the evidence, never matched to a target containing the
  net's own prior — so confidence has to be earned. `LOSS_FN` still toggles
  `'evidence'` ↔ `'kl'` for an A/B. `root_target` is still evidence-only
  (unexpanded/unproven edges contribute nothing).
- **MCTS-Solver, Thompson selection, virtual loss, opponent-pool diversity,
  multiprocess self-play + central GPU inference server, DirectML split-
  backward (lgamma/digamma have no DML kernel), checkpoint resume with the
  `weights_only` numpy-scalar fallback baked in from the start** — all
  unchanged in mechanism from the Boop notebook, just retargeted at
  `ThompsonChessNet`/chess.

### What's different, beyond the Dirichlet generalization
- **No board-symmetry augmentation.** Boop's 8× free-data multiplier doesn't
  exist for chess: a left-right mirror isn't a symmetry (King/Queen occupy
  distinct files), and the observation planes are absolute — White/Black
  pieces in fixed planes, not player-relative (there's a side-to-play plane
  instead) — so even a naive flip needs piece-plane swapping too. A genuinely
  valid symmetry *does* exist (180° rotation + swap White/Black piece planes +
  flip side-to-play — any position's value negates under full color reversal)
  but remapping the **policy target** through it requires exactly replicating
  the engine's action encoding (64 squares × 73 from-square-relative
  destination planes — see `chess.h`'s `kNumActionDestinations`); a bug there
  would silently corrupt every augmented sample with no crash. Left
  unimplemented — see "chess-specific ideas" below for what implementing it
  correctly would take.
- **Ply caps, not z-mixing.** Neither this notebook nor `bayesian_mcts_3` ever
  uses the eventual game outcome `z` in a training target — every move's
  target comes purely from *that move's own* root search belief. So
  `MAX_SELFPLAY_PLIES` (self-play) and `MAX_EVAL_PLIES` (tournaments, quick
  eval, and the arena section) are pure pacing safety valves, not correctness
  requirements: chess is only *eventually* bounded (the 50-move rule), and an
  early, undertrained or weak net can go hundreds of plies avoiding captures
  before that resets — long enough for a single evaluation game to stall a
  whole serial tournament (this is not hypothetical: it's exactly what
  happened building this notebook — a real, if slow, hang was traced to
  `run_tournament`'s and `arena()`'s unbounded `while not
  state.is_terminal()` loops during testing). Self-play games cut off early
  simply lose nothing (every recorded sample stays valid regardless of how the
  game would have ended); evaluation games cut off early score as a draw —
  which for chess is exact, not an approximation: `returns()` on a
  non-terminal chess state is defined to be exactly `[0.0, 0.0]`, chess's own
  "no result yet" value, so no special-casing was needed once the loop
  condition itself was capped.
- **Own checkpoint directory, own network class** (`ThompsonChessNet`), no
  KataZero-chess baseline to arena against (none exists — see the arena
  section for what's provided instead: checkpoint-vs-checkpoint comparisons).

### Network architecture
```
Input (20, 8, 8) — chess's native observation planes (12 piece + empty +
                    repetition-count + side-to-play + irreversible-move-
                    counter + 4 castling-rights)
                 →  Stem (Conv 3×3, GroupNorm, ReLU)
                 →  N × ResBlock (Conv-GN-ReLU-Conv-GN + SE) + skip
                 →  Head: 1×1 conv (8 ch) → flatten
                       → Linear(4674×3) → softmax(dim=-1) → (p_win,p_draw,p_loss)/action
                       → Linear(4674)   → softplus → confidence α₀ ∈ [CONF_MIN, CONF_MAX]
```
A flat Linear policy head (not a spatial from-square/destination-plane conv
head) is used for simplicity and to avoid a whole new class of indexing bugs —
see below for the spatial-head alternative as a candidate optimization.

---

## Performance engineering (AMD GPU / DirectML)

Chess's action space (4674) makes every *dense* per-action tensor ~43× bigger
than Boop's — and a careful pass over the port showed its costs were dominated
not by search or the conv body but by moving/processing those mostly-empty
dense tensors. This notebook therefore keeps dense `(B, 4674, ·)` tensors
**only inside the network forward**, and gathers down to the ~35 legal (or
~10-35 evidence) entries at every boundary:

- **Sparse training targets.** A sample is `(obs fp16, action_idx int32 (m,),
  counts fp32 (m,3))` — only the m evidence edges. Dense `(4674, 3)` fp32
  targets would be 56 KB of 99% zeros per move: ~6 GB of replay buffer at
  `MAX_BUFFER=100k` and ~20 MB of pickle traffic per episode through the
  worker queue; sparse is ~3 KB per sample all-in (~300 MB buffer), and the
  loss only ever scored evidence entries anyway.
- **Gathered loss.** Both losses now evaluate directly on the gathered
  evidence entries (`(M,3)/(M,)` instead of `(B,4674,3)/(B,4674)` + mask) —
  identical to the old dense masked mean (regression-tested against the
  original formulas) with **~200× less lgamma/digamma work**. That work runs
  on the CPU under DirectML (no DML kernels — the split-backward), so this
  was the single largest training-step cost.
- **Gathered NN responses.** Self-play workers send each request's legal-move
  indices along with the (now fp16) observations; the inference server
  responds with just the gathered `(k,3)+(k,)` legal entries — ~0.5 KB per
  leaf instead of the dense 74.8 KB row, a ~150× cut in response-queue pickle
  traffic (at 250-1000 sims/move that was tens of MB per move).
- **On-device gather, probed.** GPU→CPU readback is DirectML's most expensive
  primitive (staging-buffer copy + full pipeline sync). Where `index_select`
  works on the device — probed once at startup against known values, forward
  and backward separately, since torch-directml op coverage varies by version
  — both the server's response gather and the training loss's evidence gather
  run on-device, so only ~1% of the head outputs is ever read back. If a probe
  fails, both paths fall back to one full download + CPU gather (correct,
  just slower — and never worse than the pre-optimization behaviour).
- **Row-capped server batching** (a request is a whole wave, so the batch
  window now caps accumulated *rows*, not request count), fp16 observations on
  the wire and in the buffer (the planes are 0/1 flags plus a couple of small
  fractions — fp16 round-trips them to ~1e-3), diagnostics computed from the
  loss path's already-downloaded gathered entries (nothing dense is ever
  downloaded twice), and `legal_actions()` fetched once per eval and reused.

Already in place from the Boop notebooks (unchanged): `LerpFreeAdamW` (no
`aten::lerp` on DML), elementwise GroupNorm (the fused kernel's backward is
broken there), elementwise softplus, the model lock serializing the training
thread against the inference-server thread (DML is not thread-safe),
main-thread autograd-engine warm-up, and CPU replicas for all batch-1
inference (eval, tournaments, arena — batch-1 DML forwards are latency-bound).

Deliberately *not* done: an fp16 inference copy of the net (plausible ~2× on
RDNA, but untestable in this environment and GroupNorm's fp16 statistics are
a real risk — try it only with an A/B against fp32), and the spatial policy
head (see below — it's a *sample-efficiency* idea, not a speed fix: the head
is 76% of the parameters but only ~5% of forward FLOPs; the conv body
dominates compute).

---

## Innovation-calibrated confidence (this branch's change)

The first chess training run reproduced (much faster) the confidence-runaway
failure seen in the Boop `bayesian_mcts_3` runs: predicted α₀ pinned at
`CONF_MAX` within a few hundred episodes while strength stayed flat. This
branch adds the principled fix — **no caps, no clamps, no thresholds** — by
answering the question rigorously: *given the children's beliefs at `s`, what
posterior does the edge `a → s` deserve?*

**What the mixture backup already gets right.** With `w_i = P(action i is
best)`, the law of total variance over the unknown eventual choice gives
`E[V(s)] = Σ w_i μ_i` and `Var[V(s)] = Σ w_i σ_i²  +  Σ w_i (μ_i − μ̄)²` — and
mixing whole PMFs (as `_node_beliefs` does) computes exactly this, including
the second, **between-child disagreement** term: a sharp position whose moves
disagree wildly is correctly a position whose value is uncertain. (The fully
Bayes-consistent alternative — the distribution of the max,
`Σᵢ pᵢ(x)·Πⱼ≠ᵢ Fⱼ(x)` — is right only under *independent* child beliefs, and
sibling NN evaluations are anything but; conditioning on "looked best" then
imports the optimizer's curse. That was `bayesian_mcts` v1, rejected.)

**What sibling agreement does NOT justify.** Agreement among siblings must not
tighten the parent beyond a single child's width: the children are one smooth
function evaluated at k neighboring inputs, not independent measurements — a
badly wrong net agrees with itself, confidently, across all siblings.
Treating agreement as evidence is precisely the self-reinforcing loop that
caused the runaway. The asymmetry is real: disagreement is informative
(sharpness), agreement is not (smoothness predicts it either way).

**The missing ingredient: the innovation.** The search continuously produces
near-ground-truth evidence about belief *quality* and previously threw it
away. Just before an edge's first lookahead replacement, the edge holds the
net's direct belief `(μ_a, σ_a²)`; the one-ply-deeper value arrives as
`(μ_L, σ_L²)`. For calibrated beliefs, the innovation `e = μ_a − μ_L`
satisfies `E[e²] = σ_a² + σ_L²`, i.e. `z = e/√(σ_a²+σ_L²) ~ N(0,1)`.
Terminal/proven edges give the same measurement against the game's actual
+1/0/−1. A net claiming σ≈0.05 while lookahead moves values by ~0.5 emits
10σ innovations — its confidence is empirically refuted by data the search
already generated. (This is also the honest answer to "update the prior of
`a` with the lookahead": the *mean* is replaced — deeper dominates, and
averaging two same-NN estimates would double-count — but the *discrepancy*
is kept as calibration evidence.)

**The mechanism** (`_Calib` in the tree-ops cell): model miscalibration as a
slowly-varying variance-inflation factor λ (true error variance = λ × claimed
variance). Estimate λ as a decayed average of normalized squared innovations,
collected at every first-lookahead backup and every terminal proof; score each
observation with the bounded-influence t-statistic `ψ(z²)=(ν+1)z²/(ν+z²)`
(ν=4 — innovations are heavy-tailed even for calibrated nets; one tactical
shock must not set λ=200), normalized by its N(0,1) expectation so a
calibrated net gives **exactly λ→1**. Apply λ in one place: leaf construction
scales `Var[v]` by λ before moment-matching. P(best), Thompson sampling,
mixture propagation, and `root_target`'s reconstructed concentration then all
reflect *calibrated* uncertainty automatically.

**Why the loop now closes at honesty.** Overclaiming net → large normalized
innovations → λ rises → held beliefs widen to the observed error scale →
targets carry calibrated concentration → the evidence loss trains the net
toward them → innovations normalize → λ→1. Underconfidence runs symmetrically
(λ<1 sharpens). The old rails (`CONF_MAX`, `CONF_CAP`) remain in the code but
are expected to be non-binding: the mechanism, not a ceiling, regulates
confidence. λ is logged per episode (`lam=` in the training line, `hist['lam']`,
plotted in the plots cell): expect it well above 1 early (the net is
overconfident relative to what one ply of search reveals) and drifting toward
1 as training calibrates; λ stuck high late in training means the net's
confidence still isn't earned.

---

## Native forced-line handling: tree reuse + solver-labeled samples

The MCTS-Solver already treats proven outcomes as exact delta spikes
(infinite confidence in-tree; full `conf_cap` on the proven column in
training targets) and propagates WIN > DRAW > LOSS proofs up the tree. Two
additions make that knowledge *persist* instead of dying with each search:

- **Tree reuse across moves** (`_descend`). After a move is played, the
  subtree under it becomes the next search's root — proofs, expansions, and
  refined beliefs all carry over, so a forced mate proven at move 30 stays
  proven for every subsequent move of the game (a solved root finalizes
  instantly, spending zero simulations). This is safe *by construction* in
  this framework: beliefs are recomputed from the current subtree on every
  backup (no stale visit counts exist), and virtual losses are always unwound
  before a move is played. In opponent-pool games the kept tree is also
  descended by the frozen opponent's move to stay in sync. (Deliberately NOT
  done: a cross-game transposition table of proofs — chess proofs that rest
  on threefold repetition are history-dependent (the Graph History
  Interaction problem), so a sound cache must track repetition-freeness of
  each proof subtree; left as a follow-up.)

- **Solver-labeled auxiliary training samples.** When `_propagate_solved`
  proves an interior node, that node's `root_target` is exact game-theoretic
  ground truth — the strongest supervision this system can produce, with no
  self-reference at all. Each newly-solved node is emitted once (the emission
  point is its parent edge's transition to proven, which happens exactly once
  per node — built-in dedup) as a normal `(obs, idx, counts)` sample appended
  to the episode. Observations are stashed on nodes at expansion time
  (zero-copy: the fp16 row already exists for the NN request — which also
  lets the per-move root sample skip a redundant `observation_tensor()`
  call). The training line's `aux=` counter tracks the cumulative count:
  near zero early (weak nets rarely prove anything), growing as the net gets
  strong enough to reach forced lines — a nice progress signal in itself.

---

## Chess-specific ideas (as requested — not all implemented)

**1. Spatial policy head (real AlphaZero-style).** Chess's action space is
already structured as 64 squares × 73 "destination planes"
(`kNumActionDestinations=73` in `chess.h`, `EncodeMove`'s
`(from_square) * 73 + destination_index`) — exactly the layout AlphaZero uses.
A conv head outputting `4·73=292` channels at 8×8 resolution (4 numbers per
action: win/draw/loss logits + confidence) could replace the flat
`Linear(512, 14022)` with something that shares weights across squares
(far fewer parameters, likely better sample efficiency), by reshaping/
permuting the conv output to align with `from_square*73 + dest_idx`, plus a
small separate head for the two non-spatial castling actions
(`kLeftCastlingAction=4672`, `kRightCastlingAction=4673`). Not implemented
here: an indexing mistake in that remap would silently scramble every
policy-action correspondence (wrong gradient signal, no crash) — exactly the
class of bug this whole project has been careful to avoid elsewhere. Worth
doing, but needs its own dedicated verification pass (e.g. checking the remap
against `EncodeMove`/`ActionToMove` on real positions) before trusting it in a
training run.

**2. Color-flip 180° symmetry augmentation.** As described above: rotate the
board 180°, swap White ↔ Black piece and castling planes, flip side-to-play,
negate the value target — a real 2× data multiplier (smaller than Boop's 8×,
but chess self-play is far more expensive per game). Needs the same care as
the spatial head: verify the transformed action indices against the real
engine on constructed test positions (e.g. check that applying the mapped
action to the transformed observation reaches the transformed successor of
applying the original action to the original observation) before trusting it.

**3. History-stack input planes.** Real AlphaZero/Leela feed the last ~8
board positions as extra planes (partly to disambiguate repetition-adjacent
states beyond OpenSpiel's compressed repetition-count plane, partly because it
empirically helps). Not done here — it would mean threading a short
per-game history buffer through self-play (worker slots, the parallel
self-play loop, and `state_to_tensor`/`batch_to_tensor`), a real but bounded
extension if the compressed single-position + repetition-count encoding turns
out to be a bottleneck.

**4. Hyperparameters are a starting point, not a tuned result.** Unlike the
Boop branches (which reacted to real training curves across several rounds of
discussion), nothing here has been trained to convergence. `NUM_BLOCKS=10`,
`CHANNELS=128`, `MAX_BUFFER=100_000`, `TEMP_THRESHOLD=20`, the sim budgets,
etc. are reasonable starting points carried over from Boop's *proven*
defaults, sized up where the input/action space obviously demands it — not
validated against an actual chess training run. Expect to need substantially
more self-play episodes and wall-clock than Boop reached meaningful strength
with; watch the same telltale signs that showed up in the Boop
`bayesian_mcts_3` runs (`α₀ predicted` climbing without `α₀ target` /
strength keeping pace = overconfidence outrunning what search actually
resolved) and consider the same remedies discussed there (shrink `MAX_BUFFER`
further, cap the prior's in-tree influence, gate which checkpoints join the
benchmark/opponent pool) if training stalls or regresses.

**5. Compute cost of the Dirichlet generalization itself.** Leaf-belief
construction is now a handful of closed-form array ops instead of Boop's
single `_beta_pmf_rows` call (still `O(k)` in the number of legal actions, no
`O(G²)` convolution — that was considered and rejected in favor of the
cheaper, still-verified moment-matching construction). The mixture-propagation
backup itself is unchanged in complexity from Boop. At chess's branching
factor (~35 average legal moves vs Boop's ~10) and action space (4674 vs 108),
the dominant costs are the NN forward passes and the 64×8×8 observation
tensors, not this notebook's tree bookkeeping.

In [ ]:
%pip install open_spiel -q
# AMD/Intel GPU acceleration via DirectML (Windows / WSL2). Needs Python
# 3.11 or 3.12 (torch-directml has no 3.13 wheel) and pins torch==2.3.1.
# Harmless if you're on CUDA/CPU — the device code below falls back.
%pip install torch-directml -q

In [ ]:
import pyspiel

# Chess is already implemented natively in OpenSpiel (C++, games/chess/) — no
# custom engine needed here, unlike the Boop notebooks. It provides exact
# detection of every real chess draw (insufficient material, threefold
# repetition, stalemate, the 50-move rule) as well as checkmate, all inside
# is_terminal()/returns(); utilities are exactly WinUtility=+1, DrawUtility=0,
# LossUtility=-1, matching this notebook's v = p_win - p_loss value scale
# with no rescaling needed.
#
# Board-plane convention (see chess.cc ObservationTensor): the 20 planes are
# ABSOLUTE, not player-relative — White/Black pieces occupy fixed planes
# regardless of who is asking, with a side-to-play plane telling the network
# whose turn it is (unlike Boop's mover-relative flat observation). This is
# also why there is no board-symmetry augmentation available for chess (a
# left-right mirror is not a symmetry of the starting position: King/Queen
# occupy distinct files) — see the intro cell for a color-flip 180° rotation
# that WOULD be a valid symmetry, left unimplemented for now.
GAME = pyspiel.load_game('chess')
_NUM_ACTIONS = GAME.num_distinct_actions()
_OBS_SHAPE   = tuple(GAME.observation_tensor_shape())    # (20, 8, 8)

print('Game:', GAME.get_type().long_name)
print('Actions:', _NUM_ACTIONS)
print('Observation shape:', _OBS_SHAPE)
print('Utility (min/max):', GAME.min_utility(), GAME.max_utility())

_state = GAME.new_initial_state()
print()
print(_state)
print('Legal actions from the start position:', len(_state.legal_actions()))
del _state

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import random

# ── Device selection ──────────────────────────────────────────────────────────
# Same policy as the Boop notebooks: the GPU only wins with BIG batches, so
# self-play NN requests are funneled through one batching server and training
# runs large-batch steps there; small-batch paths (eval, arena) use CPU replicas.
DEVICE_PREFERENCE = 'directml' # 'cpu' | 'directml' | 'cuda' | 'auto'

def _pick_device(pref):
    if pref in ('directml', 'auto'):
        try:
            import torch_directml
            try:    _name = torch_directml.device_name(0)
            except Exception: _name = 'DirectML GPU'
            print(f'Using DirectML: {_name}')
            return torch_directml.device(), 'directml'
        except Exception:
            if pref == 'directml':
                print('DirectML requested but unavailable — falling back.')
    if pref in ('cuda', 'auto') and torch.cuda.is_available():
        return torch.device('cuda'), 'cuda'
    return torch.device('cpu'), 'cpu'

DEVICE, _BACKEND = _pick_device(DEVICE_PREFERENCE)


# ── Outcome convention (used EVERYWHERE in this notebook) ─────────────────────
# Chess HAS draws (insufficient material, threefold repetition, stalemate, the
# 50-move rule — all detected exactly by the C++ engine, no move-cap heuristic
# needed). So a per-action belief is a distribution over the full 3-way outcome
# (win, draw, loss) FROM THE PERSPECTIVE OF THE PLAYER TAKING THE ACTION. This
# is the Dirichlet generalization of the Boop notebooks' Beta (win/loss only):
# see the tree-ops cell for exactly how a 3-category Dirichlet belief is
# represented and mixture-propagated using the same 1D value-grid machinery.
#
# The scalar "value" used throughout for selection/comparison is
#   v = p_win − p_loss  ∈ [−1, 1],
# which is exactly chess's own utility scale (WinUtility=+1, DrawUtility=0,
# LossUtility=−1) — no rescaling needed, this framework fits chess natively.
# One ply up the tree the mover changes, so v flips sign (v → −v); a draw
# stays a draw regardless of whose turn it is (no flip for the draw share).
_WIN, _DRAW, _LOSS = 0, 1, 2                       # target columns (win, draw, loss)
_TERM_DRAW = _DRAW                                  # alias: solver/terminal draw marker
# MCTS-Solver: a proven outcome for the mover flips one ply up (win<->loss,
# draw stays), because the parent's mover is the opponent. Indexed by outcome.
_FLIP_TERM = np.array([_LOSS, _DRAW, _WIN], dtype=np.int8)

ALPHA_FLOOR = 0.05    # per-component Dirichlet floor: keeps discretization + KL finite
CONF_MIN    = 0.5     # smallest predictable prior strength (pseudo-observations)
CONF_MAX    = 100.0   # cap: stops confidence from running away across generations


# ── Input helpers ─────────────────────────────────────────────────────────────
# Chess's observation tensor is ALREADY a dense (20, 8, 8) plane stack (12
# piece planes + 1 empty plane + repetition-count + side-to-play +
# irreversible-move-counter + 4 castling-rights planes) — no reshape needed,
# unlike Boop's flat-184-float observation. It is NOT player-relative (White
# and Black pieces occupy fixed planes regardless of who is asking; a
# side-to-play plane tells the network whose turn it is), which is also why
# there is no useful board-symmetry augmentation for chess — see the intro cell.

def state_to_tensor(state, device):
    """Single game state → (1, 20, 8, 8) float tensor."""
    obs = np.array(state.observation_tensor(state.current_player()), dtype=np.float32)
    x   = obs.reshape(1, *_OBS_SHAPE)
    return torch.from_numpy(x).to(device)


def batch_to_tensor(obs_list, device):
    """List of flat observations → (B, 20, 8, 8) float tensor. Accepts python
    lists or numpy arrays of any float dtype (replay samples are stored fp16 —
    see the tree-ops cell) and converts in ONE pass."""
    obs = np.asarray(obs_list, dtype=np.float32)
    x   = obs.reshape(-1, *_OBS_SHAPE)
    return torch.from_numpy(x).to(device)


# ── Device capability probe: on-device gather (index_select) ─────────────────
# The two hottest data paths in this notebook both want to GATHER a handful of
# legal-action entries (~35 of 4674) out of the dense (B, 4674, ·) head
# outputs while they are still ON the device, so only ~1% of the data ever
# crosses the bus: (a) the self-play inference server's response rows, and
# (b) the training loss (which only scores evidence entries anyway).
# GPU→CPU readback is DirectML's single most expensive primitive (staging-
# buffer copy + full pipeline sync), so this matters far more there than on
# CUDA. torch-directml's op coverage varies by version, though — so probe
# index_select once here, against known values, forward and backward
# separately, and let each consumer fall back to a full-tensor download when
# its direction is unsupported (correct either way, just slower).
def _probe_device_gather(device):
    if str(device) == 'cpu':
        return True, True                 # plain aten ops — always available
    fwd = bwd = False
    try:
        x   = torch.arange(12.0, device=device).reshape(4, 3)
        idx = torch.tensor([2, 0, 3], dtype=torch.int64, device=device)
        y   = x.index_select(0, idx).cpu()
        fwd = bool(torch.equal(y, torch.tensor([[6., 7., 8.],
                                                [0., 1., 2.],
                                                [9., 10., 11.]])))
    except Exception:
        pass
    try:
        x   = torch.ones(4, 3, device=device, requires_grad=True)
        idx = torch.tensor([1, 1, 3], dtype=torch.int64, device=device)
        y   = x.index_select(0, idx)
        y.backward(torch.ones_like(y))    # backward of gather = index_add
        g   = x.grad.detach().cpu().sum(dim=1)
        bwd = bool(torch.allclose(g, torch.tensor([0., 6., 0., 3.])))
    except Exception:
        pass
    return fwd, bwd

_GATHER_FWD_OK, _GATHER_BWD_OK = _probe_device_gather(DEVICE)
if str(DEVICE) != 'cpu':
    print(f'on-device gather: forward '
          f'{"OK" if _GATHER_FWD_OK else "UNAVAILABLE (full-download fallback)"}'
          f', backward '
          f'{"OK" if _GATHER_BWD_OK else "UNAVAILABLE (full-download fallback)"}')


# ── Network modules (identical to the Boop notebooks) ─────────────────────────

class _GroupNorm(nn.Module):
    """GroupNorm from elementwise ops — DirectML-safe (the fused kernel's
    backward is broken there) and keeps NO running stats (train == eval)."""
    def __init__(self, num_groups, num_channels, eps=1e-5):
        super().__init__()
        self.num_groups = num_groups
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(num_channels))
        self.bias = nn.Parameter(torch.zeros(num_channels))

    def forward(self, x):
        n, c = x.shape[0], x.shape[1]
        xg = x.reshape(n, self.num_groups, -1)
        mean = xg.mean(dim=2, keepdim=True)
        var = (xg - mean).pow(2).mean(dim=2, keepdim=True)
        xg = (xg - mean) / torch.sqrt(var + self.eps)
        x = xg.reshape(x.shape)
        return x * self.weight.view(1, c, 1, 1) + self.bias.view(1, c, 1, 1)


def _norm(channels):
    groups = min(8, channels)
    while channels % groups != 0:
        groups -= 1
    return _GroupNorm(groups, channels)


def _softplus(x):
    """Numerically-stable softplus from elementary ops only. torch's fused
    aten::softplus has no DirectML kernel; relu/abs/exp/log/add all do (softmax
    already exercises exp+log on DML). Identical to F.softplus:
    log(1+e^x) = max(x,0) + log(1 + e^-|x|)."""
    return torch.relu(x) + torch.log(1.0 + torch.exp(-torch.abs(x)))


class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (KataGo-style)."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels * 2),
        )

    def forward(self, x):
        s = x.mean(dim=(2, 3))
        scale, bias = self.fc(s).chunk(2, dim=1)
        scale = torch.sigmoid(scale)
        return (x * scale[:, :, None, None]
                  + bias[:, :, None, None])


class ResBlock(nn.Module):
    """Residual block: Conv-GN-ReLU-Conv-GN + SE attention + skip."""
    def __init__(self, channels, use_se=True):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            _norm(channels),
        )
        self.se  = SEBlock(channels) if use_se else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.se(self.net(x)) + x)


class ThompsonChessNet(nn.Module):
    """
    ThompsonZero network for Chess: THREE outcome probabilities + a confidence
    per action (the Dirichlet generalization of the Boop notebooks' win-prob +
    confidence Beta head — chess has real draws, so the outcome distribution
    needs all three categories).

    Input  : (B, 20, 8, 8) — chess's native observation planes
    Body   : Conv stem → N × ResBlock(channels, SE)
    Head   : 1×1 conv (H ch) → flatten →
               dist_out Linear(H·64, NUM_ACTIONS·3) → softmax(dim=-1)
                        → (p_win, p_draw, p_loss) per action
               conf_out Linear(H·64, NUM_ACTIONS)   → softplus → α₀ ∈ [CONF_MIN, CONF_MAX]

    forward(x) → (probs (B, NUM_ACTIONS, 3), conf (B, NUM_ACTIONS)).
    The predicted prior for action a is Dirichlet(conf·probs[a]) —
    equivalently the (win, draw, loss) pseudo-counts (conf·p_win, conf·p_draw,
    conf·p_loss).

    A flat Linear policy head (not a spatial from-square/to-plane conv head)
    is used here for simplicity/robustness — see the intro cell for a
    from-square spatial head as a candidate follow-up optimization.
    """
    _HEAD_CH = 8      # wider than Boop's 4: 4674 actions vs 108 needs more head capacity

    def __init__(self, channels=128, num_blocks=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(_OBS_SHAPE[0], channels, 3, padding=1, bias=False),
            _norm(channels),
            nn.ReLU(inplace=True),
        )
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])
        self.head = nn.Sequential(
            nn.Conv2d(channels, self._HEAD_CH, 1, bias=False),
            _norm(self._HEAD_CH),
            nn.ReLU(inplace=True),
            nn.Flatten(),
        )
        flat = self._HEAD_CH * _OBS_SHAPE[1] * _OBS_SHAPE[2]
        self.dist_out = nn.Linear(flat, _NUM_ACTIONS * 3)
        self.conf_out = nn.Linear(flat, _NUM_ACTIONS)
        # Start near-uniform (p_win=p_draw=p_loss=1/3) with a weak prior
        # (α₀ ≈ 2.4): an untrained net should claim LOW confidence so search
        # observations dominate the posterior from the very first generation.
        nn.init.constant_(self.conf_out.bias, 1.4)   # CONF_MIN + softplus(1.4) ≈ 2.4

    def forward(self, x):
        h     = self.head(self.body(self.stem(x)))
        probs = F.softmax(self.dist_out(h).view(-1, _NUM_ACTIONS, 3), dim=-1)
        conf  = torch.clamp(CONF_MIN + _softplus(self.conf_out(h)), max=CONF_MAX)
        return probs, conf


print(f'Device: {DEVICE}')
_demo = ThompsonChessNet()
print(f'ThompsonChessNet params: {sum(p.numel() for p in _demo.parameters()):,}')
del _demo

In [ ]:
# ── No board-symmetry augmentation for chess ───────────────────────────────────
# The Boop notebooks got 8× free training data from board symmetries. Chess has
# NONE usable here: a left-right mirror is not a symmetry of the game (King and
# Queen occupy distinct files, so mirroring produces an illegal/different
# position), and the observation planes are absolute (White/Black pieces in
# fixed planes, not player-relative), so even a naive flip would need piece
# planes swapped too. A genuinely valid symmetry — 180° rotation + swap White
# and Black piece planes + flip side-to-play — exists (any position's value
# negates under full color reversal) but remapping the policy target through
# it requires exactly replicating the engine's action encoding (64 squares ×
# 73 from-square-relative destination planes, kNumActionDestinations=73 in
# chess.h); a bug there would silently corrupt every augmented sample. Left
# unimplemented — see the intro cell for exactly what's needed to add it later.

# ── Training-sample format (memory/bandwidth-critical) ────────────────────────
# A sample is (obs fp16 (1280,), action_idx int32 (m,), counts fp32 (m, 3)):
#   - obs is stored/shipped as float16 (the planes are 0/1 flags plus a couple
#     of small fractions like halfmove-clock/100 — fp16 round-trips them to
#     ~1e-3, far below anything the net can resolve) and converted to fp32
#     only at batch-assembly time;
#   - the target is SPARSE: only the m evidence edges (expanded or proven at
#     the root — typically 10-35 of chess's 4674 actions) with their (win,
#     draw, loss) counts. A dense (4674, 3) fp32 target would be 56 KB of 99%
#     zeros PER MOVE — 5.6 GB of replay buffer at MAX_BUFFER=100k and ~20 MB
#     of pickle traffic per episode through the worker queue; sparse is ~3 KB
#     per sample all-in and the loss only ever scored evidence entries anyway.

def augment_sample(obs_flat, act_idx, counts):
    """No-op: returns the single unaugmented (obs, action-idx, counts) sample,
    wrapped in a list for a call-site identical to the Boop notebooks'
    (`replay_buffer.extend(augment_sample(*sample))`)."""
    return [(np.asarray(obs_flat, dtype=np.float16),
             np.asarray(act_idx,  dtype=np.int32),
             np.asarray(counts,   dtype=np.float32))]


# ── Bayesian-MCTS tree: MIXTURE-propagated DIRICHLET value distributions ──────
# Each node holds, per legal action, a belief over that action's 3-way outcome
# (win, draw, loss) FROM THE MOVER'S PERSPECTIVE. Exactly as in the Boop
# bayesian_mcts_3 notebook, nothing is accumulated by counting or sampled as a
# "rollout": the belief is recomputed from the current subtree on every backup,
# and a node's value is the MIXTURE over its children weighted by each child's
# probability of being the best reply (Tesauro mixture propagation — see the
# Boop notebook's cell for the maximization-bias rationale). What's new here is
# generalizing that machinery from 2 outcomes (Beta) to 3 (Dirichlet).
#
# The key design choice: rather than track a full joint belief over the
# 2-simplex (win, draw, loss) — which would need a 2D grid and lose all of the
# clean 1D CDF-product machinery mixture propagation relies on — this tracks
# TWO independent quantities per edge, both EXACT marginals of the Dirichlet by
# its stick-breaking/aggregation property:
#   - a scalar value  v = p_win − p_loss ∈ [−1, 1]   (chess's own utility
#     scale: WinUtility=+1, DrawUtility=0, LossUtility=−1 — no rescaling
#     needed), discretized on the SAME G-point grid machinery as Boop, just
#     reinterpreted over (−1, 1) instead of (0, 1) via an affine map;
#   - a scalar mean draw-probability, mixture-propagated by the SAME P(best)
#     weights as the value (whichever child is likely to be chosen determines
#     both its value AND its draw-rate) but with NO sign flip (a draw stays a
#     draw regardless of whose move it is).
# Selection/argmax only ever needs the scalar v (the mover picks the action
# with the highest EXPECTED utility — draws contribute 0 either way), so this
# loses no information relevant to search; only the TRAINING TARGET needs the
# full 3-way reconstruction, done once at the root (see root_target).
#
# Constructing a leaf's v-belief from the NN's (p_win, p_draw, p_loss, conf)
# needs care: v = (1 − d)·(2q − 1) where d = p_draw ~ Beta(α_d, α_w+α_l) and
# q = p_win/(p_win+p_loss) ~ Beta(α_w, α_l) are INDEPENDENT Dirichlet marginals
# (the standard aggregation property — proportions within a group are
# independent of the group's total mass). This notebook computes E[v], Var[v]
# in closed form from that decomposition (verified against Monte-Carlo sampling
# from an actual Dirichlet to <1e-4 relative error across 8 parameter regimes,
# from near-uniform to extreme skew) and moment-matches a Beta to them — i.e.
# the v-belief is CONSTRUCTED with the (1−d) shrinkage baked in (a likely-draw
# position is correctly represented as having a narrower/less-decisive value
# spread), then everything downstream (mixture propagation, selection,
# MCTS-Solver, backup) reuses the Boop machinery on this v-grid UNCHANGED.
#
# Belief per edge is a length-G PMF `edge[i]` (over the unit grid `_GRID_X`,
# read out via `_GRID_V = 2·_GRID_X − 1` wherever an actual value is needed —
# mixture propagation itself is index-space-only and never needs the value
# scale) plus a scalar `draw[i]` (mean draw-probability). G is odd so index
# G//2 lands on EXACTLY v=0 (an exact "proven draw" spike) and the end indices
# are used as the "proven win/loss" spikes (Boop's existing convention).

_GRID_G  = 33
_GRID_X  = (np.arange(_GRID_G) + 0.5) / _GRID_G          # (G,) unit-interval grid
_GRID_V  = 2.0 * _GRID_X - 1.0                            # (G,) value-space read-out
_GRID_V2 = _GRID_V ** 2
_GRID_LX  = np.log(_GRID_X)
_GRID_L1X = np.log1p(-_GRID_X)
_SPIKE   = np.eye(_GRID_G)                               # _SPIKE[j] = onehot at j
_SPIKE_WIN, _SPIKE_LOSS, _SPIKE_DRAW = (_SPIKE[_GRID_G - 1], _SPIKE[0],
                                        _SPIKE[_GRID_G // 2])
_VLOSS_PEN = 1.0      # within-wave virtual-loss penalty on a sampled edge value
                       # (double Boop's 0.5: the value domain here is twice as wide)
_MAX_FRAC  = 0.0      # 0 = pure mixture propagation; 1 = hard max (bayesian_mcts)


# ── Innovation-based confidence calibration (empirical Bayes — NO caps) ───────
# The mixture backup below is the law-of-total-variance posterior for a node's
# value given its children's beliefs — including the between-child disagreement
# term (an unclear choice widens the parent's belief). What it cannot supply on
# its own is an honest SCALE for those beliefs: their widths come from the
# net's self-reported confidence, and sibling agreement is no evidence that the
# confidence is deserved (the net is one smooth function evaluated at k
# neighboring inputs — it agrees with itself whether it's right or wrong).
# Trusting the reported widths at face value is what lets confidence feed on
# itself across generations (tight priors → tight targets → tighter net).
#
# The search itself, however, produces near-independent evidence about belief
# quality on every backup, and previously threw it away: the INNOVATION. Just
# before an edge's first lookahead replacement, the edge holds the net's
# direct belief (mu_a, var_a); the one-ply-deeper value arrives as
# (mu_L, var_L). If beliefs are calibrated, e = mu_a - mu_L has
# E[e^2] = var_a + var_L, i.e. z = e/sqrt(var_a+var_L) ~ N(0,1). Terminal
# proofs give the same measurement against GROUND TRUTH (belief vs the actual
# +1/0/-1). A net claiming sigma=0.05 while lookahead moves values by 0.5 is
# emitting 10-sigma innovations — its confidence is empirically refuted by the
# search's own data.
#
# Model the miscalibration as a slowly-varying variance-inflation factor
# lambda (true error variance = lambda x claimed variance), estimate it as a
# decayed average of normalized squared innovations, and scale Var[v] by
# lambda at leaf construction. Everything downstream — P(best), Thompson
# sampling, mixture propagation, and root_target's reconstructed
# concentration — then reflects CALIBRATED uncertainty. The training loop's
# fixed point becomes honesty: an overclaiming net drives lambda up until held
# beliefs match observed innovation scale, targets carry calibrated
# concentration, the evidence loss trains the net toward them, innovations
# normalize, lambda returns to 1. Underconfidence symmetrically gives
# lambda < 1 (beliefs sharpen). No cap, clamp, or threshold anywhere.
#
# Robustness: innovations are heavy-tailed even for a calibrated net (tactical
# shocks), so each observation is scored with the bounded-influence t-statistic
# psi(z^2) = (nu+1) z^2 / (nu + z^2), normalized by its N(0,1) expectation so
# a calibrated net gives EXACTLY lambda -> 1. The decayed average with a
# modest prior at 1.0 is smoothing of a nuisance parameter, not a policy knob.

_TDOF    = 4.0                              # t dof: bounded outlier influence
_VAR_RES = (2.0 / _GRID_G) ** 2 / 12.0      # per-PMF grid-resolution variance
                                            # (a PMF can't claim tighter than
                                            # one grid cell — numerics floor)

def _psi(z2):
    return (_TDOF + 1.0) * z2 / (_TDOF + z2)

_zg = np.linspace(-10.0, 10.0, 8001)          # Riemann sum on a uniform grid
_PSI_NORM = float(np.sum(_psi(_zg ** 2)      # (np.trapz was removed in numpy 2)
                         * np.exp(-0.5 * _zg ** 2) / np.sqrt(2.0 * np.pi))
                  * (_zg[1] - _zg[0]))
del _zg


class _Calib:
    """Running empirical-Bayes estimate of the variance-inflation factor
    lambda. observe() takes a squared innovation and the beliefs' claimed
    variance sum; lam() returns the current estimate (1.0 = calibrated).
    Decayed averaging (halflife in observations) tracks the net as it trains;
    the prior (prior_n pseudo-innovations at 1.0) only steadies the estimate
    before real data arrives."""
    __slots__ = ('s', 'n', 'd')

    def __init__(self, prior_n=50.0, halflife=2000.0):
        self.s = prior_n
        self.n = prior_n
        self.d = 0.5 ** (1.0 / halflife)

    def observe(self, e2, var_sum):
        z2 = e2 / (var_sum + 2.0 * _VAR_RES)
        self.s = self.s * self.d + _psi(z2) / _PSI_NORM
        self.n = self.n * self.d + 1.0

    def lam(self):
        return self.s / self.n


def _edge_mean_var(node, idx):
    """Mean/variance of an edge's held value belief (as-held, pre-replacement)."""
    p = node.edge[idx]
    m = float(p @ _GRID_V)
    return m, max(float(p @ _GRID_V2) - m * m, 0.0)


def _expansion_innovation(node, idx, calib):
    """Record the first-lookahead innovation for a freshly expanded edge: the
    net's direct belief (still in node.edge[idx] — call BEFORE _backup
    replaces it) vs the one-ply-deeper mixture value of the new child."""
    if calib is None:
        return
    mp_, vp_ = _edge_mean_var(node, idx)
    vpmf, _ = _node_beliefs(node.children[idx])
    mc = float(vpmf @ _GRID_V)
    vc = max(float(vpmf @ _GRID_V2) - mc * mc, 0.0)
    e = mp_ - (-mc)                    # child value flips into parent's view
    calib.observe(e * e, vp_ + vc)


def _beta_pmf_rows(alpha, beta):
    """Discretize Beta(alpha_i, beta_i) onto the unit grid → (k, G) PMF rows.
    Log-space keeps large concentrations (up to CONF_MAX) finite."""
    logw = ((alpha[:, None] - 1.0) * _GRID_LX[None, :]
            + (beta[:, None] - 1.0) * _GRID_L1X[None, :])
    logw -= logw.max(axis=1, keepdims=True)
    w = np.exp(logw)
    return w / w.sum(axis=1, keepdims=True)


def _flip_pmf(pmf):
    """Reflect a value PMF around v=0 (v -> -v): equivalent to reflecting the
    unit grid around 0.5 (x_{G-1-j} = 1 - x_j), since v=2x-1 is affine."""
    return pmf[::-1].copy()


def _dirichlet_leaf_belief(alpha_w, alpha_d, alpha_l, lam=1.0):
    """NN-predicted Dirichlet(alpha_w, alpha_d, alpha_l) evidence → (v-belief
    PMF (k,G), mean draw-probability (k,)) via the exact independent (q,d)
    decomposition described above, moment-matching a Beta to the closed-form
    (E[v], Var[v]). See cell 0 / this cell's header for the derivation; the
    round-trip (construct here, reconstruct in root_target) was verified
    numerically to recover the original (p_win,p_draw,p_loss) to <2e-3 across
    8 parameter regimes. `lam` is the innovation-calibration factor (see
    _Calib): the net's CLAIMED Var[v] is scaled to the empirically observed
    error scale before the belief is built — lam > 1 widens an overconfident
    net's beliefs, lam < 1 sharpens an underconfident one's, lam = 1 (a
    calibrated net) reproduces the claim exactly."""
    C = alpha_w + alpha_d + alpha_l
    Ed = alpha_d / C
    Vd = alpha_d * (alpha_w + alpha_l) / (C**2 * (C + 1.0))
    Eq = alpha_w / (alpha_w + alpha_l)
    Vq = alpha_w * alpha_l / ((alpha_w + alpha_l)**2 * (alpha_w + alpha_l + 1.0))
    EX, VX = 1.0 - Ed, Vd
    EY, VY = 2.0 * Eq - 1.0, 4.0 * Vq
    Ev = EX * EY
    Vv = EX**2 * VY + EY**2 * VX + VX * VY
    mu01  = (Ev + 1.0) / 2.0
    var01 = np.clip(Vv / 4.0 * lam, 1e-9, None)
    var01 = np.minimum(var01, mu01 * (1.0 - mu01) * 0.999)   # below max-possible
    conc  = np.maximum(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    a_beta = np.maximum(mu01 * conc, ALPHA_FLOOR)
    b_beta = np.maximum((1.0 - mu01) * conc, ALPHA_FLOOR)
    return _beta_pmf_rows(a_beta, b_beta), Ed


def _prob_best(E, C):
    """P(edge i is the argmax) for each edge, from PMFs `E` (k,G) and their
    inclusive CDFs `C` (k,G): sum_x p_i(x) * prod_{j!=i} P(X_j < x), splitting
    ties at x half-and-half. Leave-one-out product done in log-space (stable
    when a child's CDF is ~0 far from its mass). Index-space only — identical
    to the Boop version, unaware of the value/draw interpretation."""
    logG = np.log(np.clip(C - 0.5 * E, 1e-12, 1.0))     # log P(X_j < x), tie-split
    loo  = np.exp(logG.sum(axis=0)[None, :] - logG)     # prod_{j!=i} (k, G)
    w = (E * loo).sum(axis=1)                           # (k,)
    s = w.sum()
    return w / s if s > 0 else np.full(len(w), 1.0 / len(w))


class _TNode:
    """One expanded state. `edge[i]` is the value-belief PMF for action i (from
    this node's mover's perspective); `draw[i]` is the mean draw-probability for
    action i. `term[i] >= 0` marks a PROVEN outcome for edge i (terminal state,
    or MCTS-Solver proved the subtree): _WIN, _DRAW, or _LOSS.

    Takes ALREADY-GATHERED per-legal-action rows: `p3` (k, 3) win/draw/loss
    probabilities and `conf` (k,) prior strengths for exactly this node's k
    legal actions. Gathering happens at the network boundary (locally right
    after a forward pass, or on the inference server before the response is
    queued) so the dense 4674-action head outputs never cross a process or
    device boundary — see the pool cell for the bandwidth arithmetic."""
    __slots__ = ('player', 'legal', 'edge', 'draw', 'vloss', 'term', 'children',
                 'obs')

    def __init__(self, player, legal, p3, conf, lam=1.0):
        self.player   = player
        self.legal    = np.asarray(legal, dtype=np.int32)
        self.obs      = None      # fp16 observation, stashed at expansion time
        p  = np.asarray(p3,   dtype=np.float64)
        cf = np.asarray(conf, dtype=np.float64)
        aw = np.maximum(cf * p[:, 0], ALPHA_FLOOR)
        ad = np.maximum(cf * p[:, 1], ALPHA_FLOOR)
        al = np.maximum(cf * np.clip(p[:, 2], 0.0, None), ALPHA_FLOOR)
        self.edge, self.draw = _dirichlet_leaf_belief(aw, ad, al, lam)
        self.vloss    = np.zeros(len(legal), dtype=np.int32)
        self.term     = np.full(len(legal), -1, dtype=np.int8)
        self.children = [None] * len(legal)


def _effective_edges(node):
    """Edge value-belief PMFs with terminal/proven edges replaced by spikes."""
    E = node.edge
    t = node.term
    if (t >= 0).any():
        E = E.copy()
        E[t == _WIN]  = _SPIKE_WIN
        E[t == _LOSS] = _SPIKE_LOSS
        E[t == _DRAW] = _SPIKE_DRAW
    return E


def _effective_draw(node):
    """Edge mean-draw-probabilities with terminal/proven edges overridden:
    proven win/loss -> 0 (certainly not a draw), proven draw -> 1."""
    D = node.draw
    t = node.term
    if (t >= 0).any():
        D = D.copy()
        D[t == _WIN]  = 0.0
        D[t == _LOSS] = 0.0
        D[t == _DRAW] = 1.0
    return D


def _node_beliefs(node):
    """A node's own (value PMF, mean draw-probability) = the MIXTURE over its
    edges weighted by each edge's probability of being the best reply (Tesauro
    mixture propagation), computed ONCE and reused for both quantities — the
    same weights govern "how does the node's value form" and "how does the
    node's draw-rate form", since both are properties of whichever edge ends up
    effectively chosen. Optionally blended toward the hard-max value
    distribution by `_MAX_FRAC` (draw-rate mixing is unaffected by that knob)."""
    E = _effective_edges(node)
    D = _effective_draw(node)
    if E.shape[0] == 1:
        return E[0].copy(), float(D[0])
    C = np.cumsum(E, axis=1)
    np.clip(C, 0.0, 1.0, out=C)
    w = _prob_best(E, C)                                # (k,) P(edge best)
    mix = w @ E                                          # (G,) value mixture
    mean_draw = float(w @ D)                              # scalar draw mixture
    if _MAX_FRAC > 0.0:                                  # blend toward max dist
        Fmax = C.prod(axis=0)
        mx = np.empty(_GRID_G)
        mx[0]  = Fmax[0]
        mx[1:] = np.diff(Fmax)
        np.maximum(mx, 0.0, out=mx)
        ms = mx.sum()
        mix = (1.0 - _MAX_FRAC) * mix + _MAX_FRAC * (mx / ms if ms > 0 else mix)
    s = mix.sum()
    return (mix / s if s > 0 else np.full(_GRID_G, 1.0 / _GRID_G)), mean_draw


def _sample_edge_values(node, rng):
    """One Thompson sample of each edge's value, less a virtual-loss penalty for
    in-flight edges (within-wave diversification only — beliefs are untouched).
    Draws don't need a separate sample here: v already reflects draw-risk via
    the (1-d) shrinkage baked into the value belief at construction time."""
    C = np.cumsum(_effective_edges(node), axis=1)
    u = rng.random_sample(len(node.legal))
    idx = np.minimum((C < u[:, None]).sum(axis=1), _GRID_G - 1)   # inverse-CDF
    v = _GRID_V[idx].copy()
    if node.vloss.any():
        v -= _VLOSS_PEN * node.vloss
    return v


def _select_leaf(root, root_state, rng, calib=None):
    """Descend by Thompson sampling until an unexpanded or terminal edge.
    Applies virtual loss along the path. Returns (path, leaf_state, edge):
      terminal/proven edge → (path, None, None)  — outcome in node.term
      unexpanded           → (path, state-at-leaf, (node, idx))  — needs NN eval
    path is a list of (node, action_idx). A NEWLY-discovered terminal edge is
    a ground-truth innovation for `calib`: the belief held for that edge is
    scored against the actual +1/0/-1 outcome (re-selections of already-proven
    edges record nothing — no new information)."""
    node  = root
    state = root_state.clone()
    path  = []
    while True:
        idx = int(_sample_edge_values(node, rng).argmax())
        node.vloss[idx] += 1
        path.append((node, idx))
        if node.term[idx] >= 0:
            return path, None, None
        state.apply_action(int(node.legal[idx]))
        if state.is_terminal():
            r = state.returns()[node.player]           # exactly +1 / 0 / -1
            if calib is not None:
                m0, v0 = _edge_mean_var(node, idx)
                calib.observe((m0 - r) ** 2, v0)
            node.term[idx] = _WIN if r > 0 else (_LOSS if r < 0 else _DRAW)
            return path, None, None
        child = node.children[idx]
        if child is None:
            return path, state, (node, idx)
        node = child


def _backup(path):
    """Remove the virtual loss and RECOMPUTE each expanded edge on the path from
    its child's current belief (bottom-up), so the leaf's fresh NN estimate is
    mixture-propagated to the root. Value flips sign going up a ply (opponent's
    move); draw-probability does NOT flip (a draw is a draw either way).
    Nothing is accumulated."""
    for node, idx in reversed(path):
        node.vloss[idx] -= 1
        if node.term[idx] < 0 and node.children[idx] is not None:
            vpmf, mdraw = _node_beliefs(node.children[idx])
            node.edge[idx] = _flip_pmf(vpmf)
            node.draw[idx] = mdraw


# ── MCTS-Solver: propagate proven wins/draws/losses up the tree ───────────────
# `node.term[i]` is the proven outcome of edge i for node.player. A node is
# solved once its best attainable outcome is certain: a single proven-WIN edge
# proves the node immediately; a loss/draw only once EVERY edge is proven (a
# proven DRAW beats a proven LOSS — the mover would rather force the draw). When
# a node becomes solved, the parent edge entering it is proven (flipped, since
# the parent's mover is the opponent). Proven edges use value/draw spikes and
# `_select_leaf` short-circuits them, so solved subtrees are never re-descended.
# Chess reaches genuine proven draws often (forced repetition, insufficient
# material, stalemate at the search horizon) — unlike Boop's move-cap draw,
# which was a dead rule; MCTS-Solver's draw handling is functionally important
# here, not just a corner case.

def _node_solved_outcome(node):
    """The proven outcome for node.player if the node is solved, else None. WIN
    as soon as any edge is a proven win; else DRAW if any edge is a proven draw
    and all edges are proven (the mover forces at least a draw); else LOSS only
    once every edge is a proven loss."""
    t = node.term
    if (t == _WIN).any():
        return _WIN
    if (t >= 0).all():
        return _DRAW if (t == _DRAW).any() else _LOSS
    return None


def _propagate_solved(path, aux=None, conf_cap=None):
    """Walk leaf→root; whenever a node has become fully solved, prove the parent
    edge that enters it (flipped). Stops at the first not-yet-solved ancestor or
    an already-proven parent edge.

    Solver-labeled auxiliary samples: when `aux` (a list) is given, each
    NEWLY-solved node is emitted as a training sample
    (node.obs, root_target(node, conf_cap)) — exact game-theoretic labels, the
    strongest supervision this system can produce (no self-reference at all:
    proven edges' targets are pure solver output at full conf_cap). The
    emission point is the moment the node's parent edge transitions to proven,
    which happens exactly ONCE per node (a solved node is never descended into
    again, so it can never re-solve; a second in-flight path reaching it backs
    off at the already-proven parent edge above) — built-in deduplication."""
    for k in range(len(path) - 1, 0, -1):
        node = path[k][0]
        out = _node_solved_outcome(node)
        if out is None:
            break
        parent, pidx = path[k - 1]
        if parent.term[pidx] >= 0:
            break
        parent.term[pidx] = _FLIP_TERM[out]
        if aux is not None and node.obs is not None:
            ti, tc = root_target(node, conf_cap)
            aux.append((node.obs, ti, tc))


def _backup_terminal(path, aux=None, conf_cap=None):
    """Backup for a path that ended on a proven edge: recompute beliefs up the
    path (the new terminal spike propagates via _node_beliefs), then propagate
    any newly-solved nodes up the tree (emitting solver-labeled samples into
    `aux` when provided — see _propagate_solved)."""
    _backup(path)
    _propagate_solved(path, aux, conf_cap)


def _descend(root, action):
    """Tree reuse: after `action` is played, the subtree under it becomes the
    next search's root — proofs, expansions, and refined beliefs all carry
    over, so a forced line proven once stays proven for the rest of the game.
    Safe by construction here: beliefs are recomputed from the current subtree
    on every backup (nothing depends on stale counts — there are none), and
    virtual losses are always unwound before a move is played. Returns None
    when the edge was never expanded (next search starts a fresh root)."""
    if root is None:
        return None
    hit = np.nonzero(root.legal == action)[0]
    return root.children[int(hit[0])] if len(hit) else None


def _edge_dirichlet(root):
    """Reconstruct each root edge's (p_win, p_draw, p_loss, concentration) from
    its value-belief PMF (moment-matched mean/variance of v, read out via
    _GRID_V) and its mean draw-probability — the exact inverse of
    _dirichlet_leaf_belief's forward construction (solving p_win-p_loss=E[v],
    p_win+p_loss=1-E[d], p_draw=E[d]; verified numerically to round-trip the
    original probabilities to <2e-3). Concentration comes from Var[v] alone
    (the same moment-matching Boop's `_edge_beta` used, now interpreted as the
    OVERALL resolution — a tight value estimate implies a resolved position,
    whether that resolution is toward a decisive win/loss or toward a
    well-established draw)."""
    E = _effective_edges(root)
    D = _effective_draw(root)
    mean_v = E @ _GRID_V
    var_v  = np.maximum(E @ _GRID_V2 - mean_v * mean_v, 1e-9)
    mu01   = (mean_v + 1.0) / 2.0
    var01  = np.maximum(var_v / 4.0, 1e-9)
    conc   = np.maximum(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    nondraw = np.clip(1.0 - D, 0.0, 1.0)
    p_win  = np.clip((mean_v + nondraw) / 2.0, 0.0, nondraw)
    p_loss = np.clip(nondraw - p_win, 0.0, nondraw)
    p_draw = D
    return p_win, p_draw, p_loss, conc


def root_target(root, conf_cap):
    """Root beliefs → SPARSE Dirichlet training target of EVIDENCE COUNTS:
    (action_idx int32 (m,), counts fp32 (m, 3)) over exactly the m evidence
    edges. Each edge's belief is reconstructed via `_edge_dirichlet`; proven
    edges (MCTS-Solver) are overwritten with certain, max-confidence targets
    (all mass on the proven outcome — chess has a REAL draw category, so a
    proven draw puts its confidence entirely on the draw column, unlike Boop's
    degenerate half-win-half-loss move-cap draw). Concentration is capped at
    `conf_cap` (rescaled, mean-preserving) so no single generation teaches
    unbounded certainty.

    Evidence-only: an edge carries real search evidence iff it was expanded (a
    child exists) or proven (solver/terminal). UNEXPANDED edges are just the
    net's own prior echoed back, so they are excluded — the loss then never
    scores the net against its own prior (which would hand it free confidence,
    the self-referential failure mode the evidence loss exists to avoid).
    Sparse is also exactly what the loss consumes (see the training cell): a
    dense (4674, 3) target was 56 KB of 99% zeros per move."""
    pw, pd, pl, conc = _edge_dirichlet(root)
    aw = pw * conc
    ad = pd * conc
    al = pl * conc
    t  = root.term
    aw = np.where(t == _WIN,  conf_cap,  np.where(t >= 0, ALPHA_FLOOR, aw))
    ad = np.where(t == _DRAW, conf_cap,  np.where(t >= 0, ALPHA_FLOOR, ad))
    al = np.where(t == _LOSS, conf_cap,  np.where(t >= 0, ALPHA_FLOOR, al))
    total = aw + ad + al
    scale = np.minimum(1.0, conf_cap / np.maximum(total, 1e-9))  # cap concentration
    ev = (t >= 0) | np.array([c is not None for c in root.children])
    cnt = (np.stack([aw, ad, al], axis=1) * scale[:, None])[ev]
    return root.legal[ev].astype(np.int32), cnt.astype(np.float32)


def root_pick(root, rng, thompson):
    """Final move choice: Thompson-sample the value beliefs (exploratory phase)
    or take the belief-mean argmax (deterministic endgame / evaluation)."""
    if thompson:
        v = _sample_edge_values(root, rng)
    else:
        v = _effective_edges(root) @ _GRID_V             # posterior value means
    return int(root.legal[int(v.argmax())])


# ── Batched-leaf Thompson-MCTS bot ─────────────────────────────────────────────

class ThompsonMCTSBot:
    """mcts_search(state) → root _TNode whose `.edge`/`.draw` hold the value/
    draw beliefs. Collects `batch_size` leaves per wave (virtual loss + the
    inherent randomness of Thompson sampling keep waves diverse) and evaluates
    them in one NN forward pass. Weights are read live from `network`."""

    def __init__(self, game, network, device, max_simulations,
                 batch_size=16, random_state=None, calib=None):
        self.game            = game
        self.network         = network
        self.device          = device
        self.max_simulations = max_simulations
        self.batch_size      = batch_size
        self._rng            = random_state or np.random.RandomState()
        # Innovation calibration persists ACROSS searches: miscalibration is a
        # property of the net, not of one position, so the estimate keeps
        # improving as the bot plays (see _Calib).
        self.calib           = calib if calib is not None else _Calib()

    def _eval_batch(self, states):
        obs_list = [s.observation_tensor(s.current_player()) for s in states]
        x = batch_to_tensor(obs_list, self.device)
        with torch.no_grad():
            probs, conf = self.network(x)
        return probs.cpu().numpy(), conf.cpu().numpy()   # (B,A,3), (B,A)

    def mcts_search(self, state):
        probs, conf = self._eval_batch([state])
        legal = state.legal_actions()
        root = _TNode(state.current_player(), legal,
                      probs[0][legal], conf[0][legal], lam=self.calib.lam())
        sims = 0
        while sims < self.max_simulations:
            if _node_solved_outcome(root) is not None:
                break                          # MCTS-Solver: root proven, stop
            wave    = min(self.batch_size, self.max_simulations - sims)
            pending = []                       # (path, leaf_state, (node, idx))
            for _ in range(wave):
                path, st, edge = _select_leaf(root, state, self._rng,
                                              self.calib)
                if st is None:                 # terminal/proven edge
                    _backup_terminal(path)
                    sims += 1
                else:
                    pending.append((path, st, edge))

            # Expand every unique leaf edge in ONE forward pass.
            unique = {}
            for path, st, (node, idx) in pending:
                unique.setdefault((id(node), idx), (node, idx, st))
            if unique:
                entries = list(unique.values())
                pr, cf  = self._eval_batch([st for _, _, st in entries])
                lam     = self.calib.lam()
                for (node, idx, st), p_row, c_row in zip(entries, pr, cf):
                    leg = st.legal_actions()
                    node.children[idx] = _TNode(st.current_player(), leg,
                                                p_row[leg], c_row[leg], lam)
                    # First-lookahead innovation: score the direct belief the
                    # edge still holds against the fresh one-ply-deeper value
                    # (must precede _backup, which replaces it).
                    _expansion_innovation(node, idx, self.calib)
            # Mixture-propagate each fresh leaf's estimate up its path.
            for path, st, (node, idx) in pending:
                _backup(path)
                sims += 1
        return root


def make_thompson_bot(game, network, device, num_simulations, batch_size=16):
    return ThompsonMCTSBot(game, network, device, num_simulations,
                           batch_size=batch_size)


# ── Evaluation ─────────────────────────────────────────────────────────────────

def policy_action(network, state, device, sample=False, rng=None):
    """Search-free move from the raw network: argmax of the prior-mean value
    (p_win - p_loss), or (sample=True) one Thompson sample from the predicted
    Dirichlet priors — the search-free analogue of temperature sampling."""
    with torch.no_grad():
        probs, conf = network(state_to_tensor(state, device))
    probs = probs[0].cpu().numpy()          # (A, 3)
    conf  = conf[0].cpu().numpy()
    legal = state.legal_actions()
    if sample:
        rng = rng or np.random
        a = np.maximum(conf[legal, None] * probs[legal], ALPHA_FLOOR)   # (k,3)
        g = rng.standard_gamma(a)
        p = g / (g.sum(axis=1, keepdims=True) + 1e-12)
        v = p[:, _WIN] - p[:, _LOSS]
    else:
        v = probs[legal, _WIN] - probs[legal, _LOSS]
    return legal[int(v.argmax())]


# Evaluation-side games have no equivalent of self-play's MAX_SELFPLAY_PLIES:
# chess is only EVENTUALLY bounded (the 50-move rule), and an early,
# undertrained or weak net can go hundreds of plies without a capture or pawn
# move before that resets. A single evaluation game running very long can
# stall a whole tournament/eval pulse (many games run serially). Cap it and
# score a cut-off game as a draw — the same fallback chess's own 50-move rule
# would eventually produce anyway.
MAX_EVAL_PLIES = 300


def play_match(net_a, net_b, game, n_games, device):
    """net_a vs net_b over n_games, alternating colours. net == None ==> random.
    Search-free, Thompson-sampled moves (per-game variety without extra noise).
    Returns (wins_a, draws, wins_b) from net_a's view — the quick progress
    pulse (does not touch Elo)."""
    wa = dd = wb = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2
        ply = 0
        while not state.is_terminal() and ply < MAX_EVAL_PLIES:
            net = net_a if state.current_player() == a_player else net_b
            if net is None:
                action = random.choice(state.legal_actions())
            else:
                action = policy_action(net, state, device, sample=True)
            state.apply_action(action)
            ply += 1
        if not state.is_terminal():
            dd += 1                      # ply cap hit: score as a draw
            continue
        r = state.returns()[a_player]
        if r > 0:   wa += 1
        elif r < 0: wb += 1
        else:       dd += 1
    return wa, dd, wb


def _mcts_move(bot, state):
    """Posterior-mean argmax after a full search (the in-tree Thompson sampling
    gives the per-game variety that keeps a round-robin from replaying
    identical games)."""
    return root_pick(bot.mcts_search(state), bot._rng, thompson=False)


# ── Benchmark-pool evaluation: running-Elo tournaments (no fixed anchor) ─────────
# Identical structure to the Boop notebooks: every net is a tournament player
# entering at START_ELO, updated with the Elo K-factor, in independent TRACKS
# (policy-only and one per MCTS budget) with per-track Elo tables and
# head-to-head matrices.

import math
import itertools
from collections import defaultdict


def run_tournament(players, elos, wdl, game, device,
                   games_per_pair=10, k=32.0, sims=0, opening_plies=0,
                   focus_label=None, refresh_pairs=0):
    """One full round-robin for a single track. Moves are chosen search-free
    (sims == 0: prior-mean argmax) or by Thompson-MCTS with `sims` simulations
    (posterior-mean argmax). The first `opening_plies` moves of each game are
    random (both sides) only to vary games; strength is then measured
    noise-free. Every unordered pair plays `games_per_pair` games, half per
    colour, all shuffled into random order; Elo is updated after each game.
    `elos` must already hold a rating for every label. Mutates `elos`/`wdl`."""
    net  = {p['label']: p['net'] for p in players}
    bots = {}
    if sims > 0:
        bs = max(1, min(sims, 16))
        for p in players:
            if p['net'] is not None:
                bots[p['label']] = make_thompson_bot(game, p['net'], device,
                                                     sims, bs)

    def pick(label, state):
        if net[label] is None:
            return random.choice(state.legal_actions())
        if sims <= 0:
            return policy_action(net[label], state, device, sample=False)
        return _mcts_move(bots[label], state)

    pairs = list(itertools.combinations(net.keys(), 2))
    if focus_label is not None:
        # Linear-cost mode: every pair involving the newest model, plus a few
        # random old-vs-old pairs so earlier ratings keep mixing.
        focus  = [p for p in pairs if focus_label in p]
        others = [p for p in pairs if focus_label not in p]
        random.shuffle(others)
        pairs = focus + others[:refresh_pairs]
    schedule = []
    for a, b in pairs:
        half = games_per_pair // 2
        schedule += [(a, b)] * half
        schedule += [(b, a)] * half
        schedule += [(a, b)] * (games_per_pair - 2 * half)   # odd remainder
    random.shuffle(schedule)

    for p0, p1 in schedule:
        state = game.new_initial_state()
        ply = 0
        while not state.is_terminal() and ply < MAX_EVAL_PLIES:
            if ply < opening_plies:
                action = random.choice(state.legal_actions())  # variety only
            else:
                lab = p0 if state.current_player() == 0 else p1
                action = pick(lab, state)
            state.apply_action(action)
            ply += 1
        # A non-terminal state's returns() is exactly [0.0, 0.0] in chess (the
        # engine's own "no result yet" value), so a ply-cap cutoff scores as a
        # draw here with no special-casing needed.
        r  = state.returns()[0]
        s0 = 1.0 if r > 0 else (0.0 if r < 0 else 0.5)
        e0 = 1.0 / (1.0 + 10 ** ((elos[p1] - elos[p0]) / 400.0))
        elos[p0] += k * (s0 - e0)
        elos[p1] += k * ((1.0 - s0) - (1.0 - e0))
        cell = wdl[(p0, p1)]
        if r > 0:   cell[0] += 1
        elif r < 0: cell[2] += 1
        else:       cell[1] += 1

In [ ]:
# ── Parallel self-play: many games, one forward pass ────────────────────────────
# ThompsonMCTSBot batches leaves within ONE search (≤ its batch_size). The next
# multiplier is batching ACROSS games: run N games concurrently, collect every
# game's leaf wave, and evaluate them all in a single NN forward pass — same
# orchestration as the Boop notebooks, on the Dirichlet tree ops above.
import os


def _nn_eval_states(network, device, states):
    """States → (probs (B, A, 3), conf (B, A), obs fp16 (B, 1280)) in one
    forward pass. The fp16 observations are returned so callers can stash them
    on the expanded nodes (solver-labeled samples + per-move root samples need
    them) — and evaluating from the fp16 copy means the net sees exactly the
    precision the replay buffer stores."""
    obs16 = np.asarray([s.observation_tensor(s.current_player())
                        for s in states], dtype=np.float16)
    x = batch_to_tensor(obs16, device)
    with torch.no_grad():
        probs, conf = network(x)
    return probs.cpu().numpy(), conf.cpu().numpy(), obs16


class ThompsonParallelSelfPlay:
    """Interleaves the searches of n_parallel self-play games so their leaf
    waves (and root expansions) share one NN forward pass. Weights are read
    live from `network`, so games in flight pick up training updates
    immediately (standard asynchronous self-play behaviour)."""

    def __init__(self, game, network, device, n_parallel=8, wave_per_game=8,
                 fast_sims=250, full_sims=1000, fast_prob=0.75, temp_threshold=20,
                 conf_cap=100.0, seed=None,
                 pool_prob=0.0, checkpoint_dir=None, channels=None,
                 num_blocks=None, max_selfplay_plies=400):
        self.game = game
        self.network = network
        self.device = device
        self.n_parallel = n_parallel
        self.wave = wave_per_game
        self.fast_sims = fast_sims
        self.full_sims = full_sims
        self.fast_prob = fast_prob
        self.temp_threshold = temp_threshold
        self.conf_cap = conf_cap
        # Self-play safety valve only — chess games are formally bounded (the
        # 50-move rule forces a draw eventually) but can still run very long,
        # especially with an early, undertrained net. Cutting a game off here
        # is SAFE and loses nothing: unlike KataZero, no training target in
        # this notebook depends on the eventual game outcome z (root_target is
        # built purely from that move's own root search belief) — an abandoned
        # game's already-recorded samples remain fully valid.
        self.max_selfplay_plies = max_selfplay_plies
        self._rng = np.random.RandomState(seed)
        # Opponent diversity (see _resolve_pool_moves): some fraction of games
        # have one side played by a frozen historical checkpoint instead of
        # mirror-matching the live net against itself. Only the live side's
        # moves become training samples.
        self.pool_prob = pool_prob
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}          # benchmark label -> frozen CPU ThompsonChessNet
        # Innovation calibration (see cell 5): one persistent estimator for
        # this self-play loop; its lambda scales every belief built here.
        self.calib = _Calib()
        self.last_lam = self.calib.lam()
        self.last_aux = 0             # solver-labeled samples in last episode
        self.slots = [self._new_game() for _ in range(n_parallel)]

    def _load_pool_net(self, label):
        net = self._pool_nets.get(label)
        if net is None:
            path = os.path.join(self.checkpoint_dir, f'bench_{label}.pt')
            net = ThompsonChessNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[label] = net
        return net

    def _new_game(self):
        # Playout cap randomization is chosen per game, as in serial self-play.
        sims = (self.fast_sims if self._rng.rand() < self.fast_prob
                else self.full_sims)
        slot = {'state': self.game.new_initial_state(), 'hist': [], 'aux': [],
                'move': 0, 'sims': sims, 'root': None, 'n': 0, 'pool': None}
        if self.pool_prob > 0 and self.checkpoint_dir:
            try:
                labels = [f[6:-3] for f in os.listdir(self.checkpoint_dir)
                          if f.startswith('bench_') and f.endswith('.pt')]
            except OSError:
                labels = []
            if labels and self._rng.rand() < self.pool_prob:
                label = labels[self._rng.randint(len(labels))]
                slot['pool'] = {'label': label,
                                'side': int(self._rng.randint(0, 2)),
                                'net': self._load_pool_net(label)}
        return slot

    def _resolve_pool_moves(self):
        """Resolve any pending frozen-opponent moves with a direct forward pass
        + greedy (prior-mean value argmax) choice — no search tree, since these
        moves are never training targets. Returns finished episodes (a pool move
        can end the game), matching _step()'s return contract."""
        done = []
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            probs, _c, _o = _nn_eval_states(pool['net'], 'cpu', [state])
            legal = state.legal_actions()
            v = probs[0][legal, _WIN] - probs[0][legal, _LOSS]
            action = legal[int(np.argmax(v))]
            # Keep the live side's reused tree in sync with the opponent's move.
            s['root'] = _descend(s['root'], action)
            state.apply_action(action)
            s['move'] += 1
            if state.is_terminal() or s['move'] >= self.max_selfplay_plies:
                done.append(s['hist'] + s['aux'])
                self.last_aux = len(s['aux'])
                self.slots[i] = self._new_game()
        return done

    def episodes(self):
        """Infinite generator of finished episodes: lists of (obs, target)."""
        while True:
            for data in self._step():
                self.last_lam = self.calib.lam()
                yield data

    def _step(self):
        """Advance every game by one leaf wave (one joint forward pass);
        returns any episodes that finished this step."""
        rng     = self._rng
        done    = self._resolve_pool_moves()   # frozen-opponent moves first
        pending = []            # (slot_idx, path, (node, idx)) — await NN leaf
        evals   = []            # ('root', i, state) / ('leaf', node, idx, state)
        seen    = set()         # unique leaf edges already queued for eval
        for i, s in enumerate(self.slots):
            pool = s['pool']
            if pool is not None and s['state'].current_player() == pool['side']:
                continue         # next move belongs to the frozen opponent
            if s['root'] is None:
                evals.append(('root', i, None, s['state']))
                continue         # root expands this step; leaves start next step
            if _node_solved_outcome(s['root']) is not None:
                continue         # MCTS-Solver: root proven — finalized below
            wave = min(self.wave, s['sims'] - s['n'])
            for _ in range(max(wave, 0)):
                path, st, edge = _select_leaf(s['root'], s['state'], rng,
                                              self.calib)
                if st is None:
                    _backup_terminal(path, s['aux'], self.conf_cap)
                    s['n'] += 1
                else:
                    node, idx = edge
                    pending.append((i, path, edge))
                    if (id(node), idx) not in seen:
                        seen.add((id(node), idx))
                        evals.append(('leaf', node, idx, st))

        # ONE forward pass for root expansions + every unique leaf across games.
        if evals:
            probs, conf, obs16 = _nn_eval_states(self.network, self.device,
                                                 [e[3] for e in evals])
            lam = self.calib.lam()
            for (kind, a, b, st), p_row, c_row, o_row in zip(evals, probs,
                                                             conf, obs16):
                leg = st.legal_actions()
                child = _TNode(st.current_player(), leg,
                               p_row[leg], c_row[leg], lam)
                child.obs = o_row
                if kind == 'root':
                    self.slots[a]['root'] = child
                else:
                    a.children[b] = child
                    # First-lookahead innovation for the freshly expanded edge
                    # (before the _backup pass replaces its held belief).
                    _expansion_innovation(a, b, self.calib)
        for i, path, (node, idx) in pending:
            _backup(path)                       # mixture-propagate the fresh leaf
            self.slots[i]['n'] += 1

        # Finalize finished (or solver-proven, or ply-capped) searches; emit
        # completed episodes.
        for i, s in enumerate(self.slots):
            if s['root'] is None:
                continue
            if s['n'] < s['sims'] and _node_solved_outcome(s['root']) is None:
                continue
            self._play_move(i)
            if s['state'].is_terminal() or s['move'] >= self.max_selfplay_plies:
                done.append(s['hist'] + s['aux'])
                self.last_aux = len(s['aux'])
                self.slots[i] = self._new_game()
        return done

    def _play_move(self, i):
        """Turn a finished search into a training sample + a move: the target is
        the root's per-action (win, draw, loss) belief moment-matched to a
        Dirichlet (solver-adjusted for proven edges); the move is Thompson-
        sampled from those beliefs (near-greedy belief mean after
        temp_threshold)."""
        s     = self.slots[i]
        state = s['state']
        root  = s['root']
        obs   = (root.obs if root.obs is not None else
                 np.asarray(state.observation_tensor(state.current_player()),
                            dtype=np.float16))    # fp16 storage (see cell 5)
        ti, tc = root_target(root, self.conf_cap)
        s['hist'].append((obs, ti, tc))
        action = root_pick(root, self._rng,
                           thompson=(s['move'] < self.temp_threshold))
        # Tree reuse: keep the played move's subtree as the next root.
        s['root'] = _descend(root, action)
        state.apply_action(action)
        s['move'] += 1
        s['n']     = 0

In [ ]:
# ── Multiprocessing self-play + central GPU inference server ─────────────────
# Same design as the Boop notebooks: N CPU worker processes do ALL game and
# tree work (pure Python — processes sidestep the GIL) and ship their NN
# requests to ONE server thread here, which batches requests from all workers
# into single large forward passes on the GPU. Small-batch inference (eval
# games, tournaments, arena) runs on CPU replicas elsewhere.
#
# Windows/Jupyter constraint: spawned processes cannot pickle notebook-cell
# functions, so the worker code is written to a real module file first.
import pathlib

_WORKER_SRC = r'''# AUTO-GENERATED by chess_thompson_zero_training.ipynb — do not edit by hand.
# Self-play worker module: Dirichlet-MCTS tree ops + worker loop. Loads chess
# directly (pyspiel.load_game('chess')) — no custom engine to embed, unlike
# the Boop notebooks. Exists as a real file because Windows multiprocessing
# (spawn) cannot pickle functions defined inside notebook cells.

# ═══════════════════════════════════════════════════════════════════════════════
# Bayesian-MCTS tree ops — standalone copy of the notebook's semantics (Dirichlet
# value/draw beliefs per edge, mixture-propagation, distribution backup,
# MCTS-Solver). Workers are pure CPU: no torch import; every NN evaluation is
# shipped to the main process's GPU inference server. Chess itself needs no
# custom engine here (unlike the Boop notebooks) — it's OpenSpiel's native C++
# game, loaded the same way the main process does.
# ═══════════════════════════════════════════════════════════════════════════════
import os
import numpy as np
import pyspiel

_GAME = pyspiel.load_game('chess')
_NUM_ACTIONS = _GAME.num_distinct_actions()

_WIN, _DRAW, _LOSS = 0, 1, 2
_FLIP_TERM = np.array([_LOSS, _DRAW, _WIN], dtype=np.int8)
ALPHA_FLOOR = 0.05

_GRID_G  = 33
_GRID_X  = (np.arange(_GRID_G) + 0.5) / _GRID_G
_GRID_V  = 2.0 * _GRID_X - 1.0
_GRID_V2 = _GRID_V ** 2
_GRID_LX  = np.log(_GRID_X)
_GRID_L1X = np.log1p(-_GRID_X)
_SPIKE   = np.eye(_GRID_G)
_SPIKE_WIN, _SPIKE_LOSS, _SPIKE_DRAW = (_SPIKE[_GRID_G - 1], _SPIKE[0],
                                        _SPIKE[_GRID_G // 2])
_VLOSS_PEN = 1.0
_MAX_FRAC  = 0.0      # 0 = pure mixture propagation; 1 = hard max


# Innovation-based confidence calibration — see the tree-ops cell for the full
# derivation. lambda = empirically-estimated variance-inflation of the net's
# claimed confidence; estimated from backup innovations, applied at leaf
# construction. No caps/clamps: a calibrated net gives lambda -> 1 exactly.
_TDOF    = 4.0
_VAR_RES = (2.0 / _GRID_G) ** 2 / 12.0

def _psi(z2):
    return (_TDOF + 1.0) * z2 / (_TDOF + z2)

_zg = np.linspace(-10.0, 10.0, 8001)          # Riemann sum on a uniform grid
_PSI_NORM = float(np.sum(_psi(_zg ** 2)      # (np.trapz was removed in numpy 2)
                         * np.exp(-0.5 * _zg ** 2) / np.sqrt(2.0 * np.pi))
                  * (_zg[1] - _zg[0]))
del _zg


class _Calib:
    __slots__ = ('s', 'n', 'd')

    def __init__(self, prior_n=50.0, halflife=2000.0):
        self.s = prior_n
        self.n = prior_n
        self.d = 0.5 ** (1.0 / halflife)

    def observe(self, e2, var_sum):
        z2 = e2 / (var_sum + 2.0 * _VAR_RES)
        self.s = self.s * self.d + _psi(z2) / _PSI_NORM
        self.n = self.n * self.d + 1.0

    def lam(self):
        return self.s / self.n


def _edge_mean_var(node, idx):
    p = node.edge[idx]
    m = float(p @ _GRID_V)
    return m, max(float(p @ _GRID_V2) - m * m, 0.0)


def _expansion_innovation(node, idx, calib):
    if calib is None:
        return
    mp_, vp_ = _edge_mean_var(node, idx)
    vpmf, _ = _node_beliefs(node.children[idx])
    mc = float(vpmf @ _GRID_V)
    vc = max(float(vpmf @ _GRID_V2) - mc * mc, 0.0)
    e = mp_ - (-mc)
    calib.observe(e * e, vp_ + vc)


def _beta_pmf_rows(alpha, beta):
    logw = ((alpha[:, None] - 1.0) * _GRID_LX[None, :]
            + (beta[:, None] - 1.0) * _GRID_L1X[None, :])
    logw -= logw.max(axis=1, keepdims=True)
    w = np.exp(logw)
    return w / w.sum(axis=1, keepdims=True)


def _flip_pmf(pmf):
    return pmf[::-1].copy()


def _dirichlet_leaf_belief(alpha_w, alpha_d, alpha_l, lam=1.0):
    C = alpha_w + alpha_d + alpha_l
    Ed = alpha_d / C
    Vd = alpha_d * (alpha_w + alpha_l) / (C**2 * (C + 1.0))
    Eq = alpha_w / (alpha_w + alpha_l)
    Vq = alpha_w * alpha_l / ((alpha_w + alpha_l)**2 * (alpha_w + alpha_l + 1.0))
    EX, VX = 1.0 - Ed, Vd
    EY, VY = 2.0 * Eq - 1.0, 4.0 * Vq
    Ev = EX * EY
    Vv = EX**2 * VY + EY**2 * VX + VX * VY
    mu01  = (Ev + 1.0) / 2.0
    var01 = np.clip(Vv / 4.0 * lam, 1e-9, None)
    var01 = np.minimum(var01, mu01 * (1.0 - mu01) * 0.999)
    conc  = np.maximum(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    a_beta = np.maximum(mu01 * conc, ALPHA_FLOOR)
    b_beta = np.maximum((1.0 - mu01) * conc, ALPHA_FLOOR)
    return _beta_pmf_rows(a_beta, b_beta), Ed


def _prob_best(E, C):
    logG = np.log(np.clip(C - 0.5 * E, 1e-12, 1.0))
    loo  = np.exp(logG.sum(axis=0)[None, :] - logG)
    w = (E * loo).sum(axis=1)
    s = w.sum()
    return w / s if s > 0 else np.full(len(w), 1.0 / len(w))


class _TNode:
    # Takes ALREADY-GATHERED per-legal-action rows: p3 (k, 3) win/draw/loss
    # probabilities + conf (k,) — the server gathers legal entries out of the
    # dense head outputs before responding (74.8 KB -> ~0.5 KB per leaf on the
    # response queue; see the pool cell).
    __slots__ = ('player', 'legal', 'edge', 'draw', 'vloss', 'term', 'children',
                 'obs')

    def __init__(self, player, legal, p3, conf, lam=1.0):
        self.player   = player
        self.legal    = np.asarray(legal, dtype=np.int32)
        self.obs      = None      # fp16 observation, stashed at expansion time
        p  = np.asarray(p3,   dtype=np.float64)
        cf = np.asarray(conf, dtype=np.float64)
        aw = np.maximum(cf * p[:, 0], ALPHA_FLOOR)
        ad = np.maximum(cf * p[:, 1], ALPHA_FLOOR)
        al = np.maximum(cf * np.clip(p[:, 2], 0.0, None), ALPHA_FLOOR)
        self.edge, self.draw = _dirichlet_leaf_belief(aw, ad, al, lam)
        self.vloss    = np.zeros(len(legal), dtype=np.int32)
        self.term     = np.full(len(legal), -1, dtype=np.int8)
        self.children = [None] * len(legal)


def _effective_edges(node):
    E = node.edge
    t = node.term
    if (t >= 0).any():
        E = E.copy()
        E[t == _WIN]  = _SPIKE_WIN
        E[t == _LOSS] = _SPIKE_LOSS
        E[t == _DRAW] = _SPIKE_DRAW
    return E


def _effective_draw(node):
    D = node.draw
    t = node.term
    if (t >= 0).any():
        D = D.copy()
        D[t == _WIN]  = 0.0
        D[t == _LOSS] = 0.0
        D[t == _DRAW] = 1.0
    return D


def _node_beliefs(node):
    E = _effective_edges(node)
    D = _effective_draw(node)
    if E.shape[0] == 1:
        return E[0].copy(), float(D[0])
    C = np.cumsum(E, axis=1)
    np.clip(C, 0.0, 1.0, out=C)
    w = _prob_best(E, C)
    mix = w @ E
    mean_draw = float(w @ D)
    if _MAX_FRAC > 0.0:
        Fmax = C.prod(axis=0)
        mx = np.empty(_GRID_G)
        mx[0]  = Fmax[0]
        mx[1:] = np.diff(Fmax)
        np.maximum(mx, 0.0, out=mx)
        ms = mx.sum()
        mix = (1.0 - _MAX_FRAC) * mix + _MAX_FRAC * (mx / ms if ms > 0 else mix)
    s = mix.sum()
    return (mix / s if s > 0 else np.full(_GRID_G, 1.0 / _GRID_G)), mean_draw


def _sample_edge_values(node, rng):
    C = np.cumsum(_effective_edges(node), axis=1)
    u = rng.random_sample(len(node.legal))
    idx = np.minimum((C < u[:, None]).sum(axis=1), _GRID_G - 1)
    v = _GRID_V[idx].copy()
    if node.vloss.any():
        v -= _VLOSS_PEN * node.vloss
    return v


def _select_leaf(root, root_state, rng, calib=None):
    node  = root
    state = root_state.clone()
    path  = []
    while True:
        idx = int(_sample_edge_values(node, rng).argmax())
        node.vloss[idx] += 1
        path.append((node, idx))
        if node.term[idx] >= 0:
            return path, None, None
        state.apply_action(int(node.legal[idx]))
        if state.is_terminal():
            r = state.returns()[node.player]
            if calib is not None:                 # ground-truth innovation
                m0, v0 = _edge_mean_var(node, idx)
                calib.observe((m0 - r) ** 2, v0)
            node.term[idx] = _WIN if r > 0 else (_LOSS if r < 0 else _DRAW)
            return path, None, None
        child = node.children[idx]
        if child is None:
            return path, state, (node, idx)
        node = child


def _backup(path):
    for node, idx in reversed(path):
        node.vloss[idx] -= 1
        if node.term[idx] < 0 and node.children[idx] is not None:
            vpmf, mdraw = _node_beliefs(node.children[idx])
            node.edge[idx] = _flip_pmf(vpmf)
            node.draw[idx] = mdraw


def _node_solved_outcome(node):
    """Proven outcome for node.player if solved, else None (MCTS-Solver): WIN
    as soon as any edge is a proven win; else DRAW if any edge is a proven draw
    and all edges are proven; LOSS only once every edge is a proven loss."""
    t = node.term
    if (t == _WIN).any():
        return _WIN
    if (t >= 0).all():
        return _DRAW if (t == _DRAW).any() else _LOSS
    return None


def _propagate_solved(path, aux=None, conf_cap=None):
    # Emits each NEWLY-solved node once as a solver-labeled sample into `aux`
    # (exact game-theoretic labels) — see the tree-ops cell for the argument.
    for k in range(len(path) - 1, 0, -1):
        node = path[k][0]
        out = _node_solved_outcome(node)
        if out is None:
            break
        parent, pidx = path[k - 1]
        if parent.term[pidx] >= 0:
            break
        parent.term[pidx] = _FLIP_TERM[out]
        if aux is not None and node.obs is not None:
            ti, tc = root_target(node, conf_cap)
            aux.append((node.obs, ti, tc))


def _backup_terminal(path, aux=None, conf_cap=None):
    _backup(path)
    _propagate_solved(path, aux, conf_cap)


def _descend(root, action):
    # Tree reuse: the played action's subtree becomes the next search's root
    # (proofs and refined beliefs carry over). None if never expanded.
    if root is None:
        return None
    hit = np.nonzero(root.legal == action)[0]
    return root.children[int(hit[0])] if len(hit) else None


def _edge_dirichlet(root):
    E = _effective_edges(root)
    D = _effective_draw(root)
    mean_v = E @ _GRID_V
    var_v  = np.maximum(E @ _GRID_V2 - mean_v * mean_v, 1e-9)
    mu01   = (mean_v + 1.0) / 2.0
    var01  = np.maximum(var_v / 4.0, 1e-9)
    conc   = np.maximum(mu01 * (1.0 - mu01) / var01 - 1.0, 2.0 * ALPHA_FLOOR)
    nondraw = np.clip(1.0 - D, 0.0, 1.0)
    p_win  = np.clip((mean_v + nondraw) / 2.0, 0.0, nondraw)
    p_loss = np.clip(nondraw - p_win, 0.0, nondraw)
    p_draw = D
    return p_win, p_draw, p_loss, conc


def root_target(root, conf_cap):
    # SPARSE target: (action_idx int32 (m,), counts fp32 (m, 3)) over exactly
    # the m evidence edges (expanded or proven) — see the tree-ops cell.
    pw, pd, pl, conc = _edge_dirichlet(root)
    aw = pw * conc
    ad = pd * conc
    al = pl * conc
    t  = root.term
    aw = np.where(t == _WIN,  conf_cap, np.where(t >= 0, ALPHA_FLOOR, aw))
    ad = np.where(t == _DRAW, conf_cap, np.where(t >= 0, ALPHA_FLOOR, ad))
    al = np.where(t == _LOSS, conf_cap, np.where(t >= 0, ALPHA_FLOOR, al))
    total = aw + ad + al
    scale = np.minimum(1.0, conf_cap / np.maximum(total, 1e-9))
    ev = (t >= 0) | np.array([c is not None for c in root.children])
    cnt = (np.stack([aw, ad, al], axis=1) * scale[:, None])[ev]
    return root.legal[ev].astype(np.int32), cnt.astype(np.float32)


def root_pick(root, rng, thompson):
    if thompson:
        v = _sample_edge_values(root, rng)
    else:
        v = _effective_edges(root) @ _GRID_V
    return int(root.legal[int(v.argmax())])


def run_worker(worker_id, req_q, resp_q, pool_resp_q, episode_q, cfg):
    """Pipelined self-play worker (leapfrog halves, same design as the Boop
    notebooks' worker): while one half's NN request is in flight on the
    central server, the CPU does tree work for the other half. NN requests are
    tagged 'live' (the training net) or a benchmark label (a frozen opponent-
    diversity net); the server routes them and answers 'live' on resp_q, pool
    nets on pool_resp_q.

    Wire format (bandwidth-critical — every leaf eval crosses two pickling
    process queues): requests carry fp16 observations PLUS each row's legal-
    action indices, `(worker_id, net_id, obs (n, 1280) fp16, legals [int32])`;
    responses carry ONLY the gathered legal entries per row,
    `[(p3 (k, 3) fp32, conf (k,) fp32), ...]` — ~0.5 KB instead of the 74.8 KB
    dense (4674, 3)+(4674,) rows."""
    rng  = np.random.RandomState(cfg['seed'] + worker_id * 7919)
    game = _GAME
    conf_cap  = cfg['conf_cap']
    pool_prob = cfg.get('pool_prob', 0.0)
    pool_dir  = cfg.get('checkpoint_dir')
    max_plies = cfg.get('max_selfplay_plies', 400)
    # One calibrator per worker, persistent across games: miscalibration is a
    # property of the (continuously-updating) net, and the decayed average
    # tracks it. Its lambda ships back with every episode for logging.
    calib = _Calib()

    def new_game():
        sims = (cfg['fast_sims'] if rng.rand() < cfg['fast_prob']
                else cfg['full_sims'])
        slot = {'state': game.new_initial_state(), 'hist': [], 'aux': [],
                'move': 0, 'sims': sims, 'root': None, 'n': 0, 'pool': None}
        # Opponent diversity: some games have one side played by a FROZEN
        # historical checkpoint instead of the live net mirror-matching itself.
        # Only the live side's moves feed the replay buffer.
        if pool_prob > 0 and pool_dir:
            try:
                labels = [f[6:-3] for f in os.listdir(pool_dir)
                          if f.startswith('bench_') and f.endswith('.pt')]
            except OSError:
                labels = []
            if labels and rng.rand() < pool_prob:
                slot['pool'] = {'label': labels[rng.randint(len(labels))],
                                'side': int(rng.randint(0, 2))}
        return slot

    slots = [new_game() for _ in range(cfg['games_per_worker'])]
    mid = max(1, cfg['games_per_worker'] // 2)
    halves = [list(range(mid)), list(range(mid, cfg['games_per_worker']))]

    def collect(idxs):
        """Gather root expansions + one leaf wave per game. Terminal edges are
        backed up immediately; pool-opponent turns and solver-proven roots are
        skipped. Returns (evals, paths, obs, legals): `evals` are the unique NN
        jobs (obs/legals rows aligned with them, legal actions fetched exactly
        once and reused at apply time), `paths` every pending leaf awaiting an
        outcome."""
        evals, paths, obs, legals = [], [], [], []
        seen = set()
        for i in idxs:
            s = slots[i]
            st0 = s['state']
            pool = s['pool']
            if pool is not None and st0.current_player() == pool['side']:
                continue         # next move belongs to the frozen opponent
            if s['root'] is None:
                leg = st0.legal_actions()
                o = np.asarray(st0.observation_tensor(st0.current_player()),
                               dtype=np.float16)
                evals.append(('root', i, leg, o))
                obs.append(o)
                legals.append(np.asarray(leg, dtype=np.int32))
                continue
            if _node_solved_outcome(s['root']) is not None:
                continue         # MCTS-Solver: root proven — finalized below
            wave = min(cfg['wave'], s['sims'] - s['n'])
            for _ in range(max(wave, 0)):
                path, st, edge = _select_leaf(s['root'], st0, rng, calib)
                if st is None:
                    _backup_terminal(path, s['aux'], conf_cap)
                    s['n'] += 1
                    continue
                node, idx = edge
                paths.append((i, path, node, idx))
                if (id(node), idx) not in seen:
                    seen.add((id(node), idx))
                    leg = st.legal_actions()
                    o = np.asarray(
                        st.observation_tensor(st.current_player()),
                        dtype=np.float16)
                    evals.append(('leaf', node, idx, st.current_player(), leg,
                                  o))
                    obs.append(o)
                    legals.append(np.asarray(leg, dtype=np.int32))
        return evals, paths, obs, legals

    def apply_and_advance(idxs, evals, paths, resp):
        """Expand a completed wave (roots + leaves) from the server's gathered
        response rows, draw and back up leaf outcomes, then finish any
        completed (or solver-proven, or ply-capped) searches: record the
        posterior target and play the Thompson-sampled move."""
        if evals:
            lam = calib.lam()
            for e, (p3, c) in zip(evals, resp):
                if e[0] == 'root':
                    _, i, leg, o = e
                    st = slots[i]['state']
                    nd = _TNode(st.current_player(), leg, p3, c, lam)
                    nd.obs = o
                    slots[i]['root'] = nd
                else:
                    _, node, idx, player, leg, o = e
                    nd = _TNode(player, leg, p3, c, lam)
                    nd.obs = o
                    node.children[idx] = nd
                    # First-lookahead innovation: score the direct belief this
                    # edge still holds against the one-ply-deeper value (must
                    # precede the _backup pass below, which replaces it).
                    _expansion_innovation(node, idx, calib)
        for i, path, node, idx in paths:
            _backup(path)                       # mixture-propagate the fresh leaf
            slots[i]['n'] += 1
        for i in idxs:
            s = slots[i]
            if s['root'] is None:
                continue
            if s['n'] < s['sims'] and _node_solved_outcome(s['root']) is None:
                continue
            state = s['state']
            root = s['root']
            o = (root.obs if root.obs is not None else
                 np.asarray(state.observation_tensor(state.current_player()),
                            dtype=np.float16))
            ti, tc = root_target(root, conf_cap)
            s['hist'].append((o, ti, tc))
            action = root_pick(root, rng,
                               thompson=(s['move'] < cfg['temp_threshold']))
            # Tree reuse: keep the played move's subtree as the next root.
            s['root'] = _descend(root, action)
            state.apply_action(action)
            s['move'] += 1
            s['n'] = 0
            if state.is_terminal() or s['move'] >= max_plies:
                episode_q.put((s['hist'] + s['aux'], calib.lam(),
                               len(s['aux'])))
                slots[i] = new_game()

    def resolve_pool_moves(idxs):
        """For any game whose CURRENT mover is the frozen-opponent seat, resolve
        that single move with ONE direct request to the pool net (tagged by its
        benchmark label) and a greedy value argmax — no search tree, since these
        moves are never training targets. Turns strictly alternate, so at most
        one such move is pending per game here."""
        for i in idxs:
            s = slots[i]
            pool = s['pool']
            if pool is None:
                continue
            state = s['state']
            if state.current_player() != pool['side']:
                continue
            obs = np.asarray(state.observation_tensor(state.current_player()),
                             dtype=np.float16)
            legal = state.legal_actions()
            req_q.put((worker_id, pool['label'], obs[None],
                       [np.asarray(legal, dtype=np.int32)]))
            (p3, _c), = pool_resp_q.get()       # one gathered row back
            v = p3[:, _WIN] - p3[:, _LOSS]
            action = legal[int(np.argmax(v))]
            # Keep the live side's reused tree in sync with the opponent's move.
            s['root'] = _descend(s['root'], action)
            state.apply_action(action)
            s['move'] += 1
            if state.is_terminal() or s['move'] >= max_plies:
                episode_q.put((s['hist'] + s['aux'], calib.lam(),
                               len(s['aux'])))
                slots[i] = new_game()

    # Leapfrog loop. Send/receive alternate A,B,A,B so the per-worker FIFO
    # response queue always delivers the response for the half we expect;
    # the `sent` flag keeps alignment when a half had nothing to evaluate.
    inflight = [None, None]          # per half: (evals, paths, sent)
    while True:
        for h in (0, 1):
            if inflight[h] is not None:
                evals, paths, sent = inflight[h]
                resp = resp_q.get() if sent else None
                apply_and_advance(halves[h], evals, paths, resp)
                inflight[h] = None
            resolve_pool_moves(halves[h])
            evals, paths, obs, legals = collect(halves[h])
            sent = False
            if evals:
                req_q.put((worker_id, 'live', np.stack(obs), legals))
                sent = True
            inflight[h] = (evals, paths, sent)
'''

pathlib.Path('chess_thompson_worker.py').write_text(_WORKER_SRC, encoding='utf-8')

import multiprocessing as _mp
import threading
import queue as _queue
import time as _time
import os as _os
import sys as _sys


class MPSelfPlayPool:
    """n_workers CPU processes play games; an inference-server thread in this
    process batches ALL their NN requests into single forward passes on
    `device`. Exposes the same .episodes() generator as ThompsonParallelSelfPlay.
    `lock` serializes model access between the server thread and training."""

    def __init__(self, network, device, n_workers, cfg, batch_window_s=0.002,
                 checkpoint_dir=None, channels=None, num_blocks=None):
        self.network = network
        self.device = device
        self.checkpoint_dir = checkpoint_dir
        self.channels = channels
        self.num_blocks = num_blocks
        self._pool_nets = {}          # benchmark label -> frozen CPU ThompsonChessNet
        self.lock = threading.Lock()
        self._stop = threading.Event()
        ctx = _mp.get_context('spawn')            # Windows-compatible
        self.req_q = ctx.Queue()
        self.episode_q = ctx.Queue(maxsize=64)    # backpressure on workers
        self.resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.pool_resp_qs = [ctx.Queue() for _ in range(n_workers)]
        self.window = batch_window_s
        # Initialize the autograd engine's per-device state from the MAIN
        # thread before any other thread touches the device: otherwise the
        # first backward races the server thread's device use and trips
        # torch-directml's "device_ready_queues_" INTERNAL ASSERT.
        if str(device) != 'cpu':
            _t = torch.zeros(4, device=device, requires_grad=True)
            (_t * 2.0).sum().backward()
        if _os.getcwd() not in _sys.path:
            _sys.path.insert(0, _os.getcwd())
        import chess_thompson_worker
        self.procs = [ctx.Process(target=chess_thompson_worker.run_worker,
                                  args=(i, self.req_q, self.resp_qs[i],
                                        self.pool_resp_qs[i],
                                        self.episode_q, cfg), daemon=True)
                      for i in range(n_workers)]
        for p in self.procs:
            p.start()
        self.server = threading.Thread(target=self._serve, daemon=True)
        self.server.start()

    def _get_net(self, net_id):
        """'live' -> the training network on its real device (needs the lock:
        DirectML thread-safety + training mutates its weights concurrently).
        Any other net_id -> a frozen benchmark checkpoint, lazily loaded once
        and cached, always run on CPU (no lock: never mutated). Pool batches are
        tiny and infrequent, so CPU inference for them costs nothing worth
        optimizing."""
        if net_id == 'live':
            return self.network, self.device, True
        net = self._pool_nets.get(net_id)
        if net is None:
            path = _os.path.join(self.checkpoint_dir, f'bench_{net_id}.pt')
            net = ThompsonChessNet(self.channels, self.num_blocks)
            net.load_state_dict(torch.load(path, map_location='cpu',
                                           weights_only=True))
            net.eval()
            self._pool_nets[net_id] = net
        return net, 'cpu', False

    def _forward_gathered(self, net, net_device, xin, flat):
        """One forward pass + gather of the legal-action entries. `flat` holds
        int64 indices into the flattened (B·A) action axis. When the device
        supports index_select (probed in the net cell), the gather runs
        ON-DEVICE and only the selected ~1% of the head outputs is read back —
        GPU→CPU readback (staging copy + pipeline sync) is DirectML's most
        expensive primitive, and the dense outputs here are
        (B, 4674, 3) + (B, 4674) fp32 ≈ 75 KB per leaf. Fallback: read the
        dense tensors back once and gather with numpy (correct, just slower)."""
        x = torch.from_numpy(xin).to(net_device)
        probs, conf = net(x)
        if _GATHER_FWD_OK and str(net_device) != 'cpu':
            ft = torch.from_numpy(flat).to(net_device)
            p_sel = probs.reshape(-1, 3).index_select(0, ft).cpu().numpy()
            c_sel = conf.reshape(-1).index_select(0, ft).cpu().numpy()
        else:
            p_sel = probs.reshape(-1, 3).cpu().numpy()[flat]
            c_sel = conf.reshape(-1).cpu().numpy()[flat]
        return p_sel, c_sel

    def _serve(self):
        A = _NUM_ACTIONS
        while not self._stop.is_set():
            try:
                reqs = [self.req_q.get(timeout=0.1)]
            except _queue.Empty:
                continue
            # Short batching window: gather concurrent workers' requests so
            # the GPU sees one big batch instead of several small ones. Capped
            # by accumulated ROWS (leaf evals), not request count — one
            # request can be a whole wave of games_per_worker × wave rows.
            rows = reqs[0][2].shape[0]
            deadline = _time.monotonic() + self.window
            while _time.monotonic() < deadline and rows < 1024:
                try:
                    r = self.req_q.get_nowait()
                    reqs.append(r)
                    rows += r[2].shape[0]
                except _queue.Empty:
                    _time.sleep(0.0003)
            # Group by net_id: almost always all 'live', occasionally a request
            # or two tagged with a frozen benchmark label (opponent-diversity
            # games) mixed in.
            groups = {}
            for wid, net_id, obs, legals in reqs:
                groups.setdefault(net_id, []).append((wid, obs, legals))
            for net_id, group in groups.items():
                net, net_device, needs_lock = self._get_net(net_id)
                obs = np.concatenate([o for _, o, _ in group], axis=0)
                xin = obs.reshape(-1, *_OBS_SHAPE).astype(np.float32)  # fp16 wire
                row_legals = [l for _, _, ls in group for l in ls]
                flat = np.concatenate([l.astype(np.int64) + r * A
                                       for r, l in enumerate(row_legals)])
                offs = np.zeros(len(row_legals) + 1, dtype=np.int64)
                np.cumsum([len(l) for l in row_legals], out=offs[1:])
                # EVERY device op on the LIVE net under the lock (upload,
                # forward, gather, download): DirectML is not safe under
                # concurrent access from two threads. CPU pool nets: no lock.
                if needs_lock:
                    with self.lock, torch.no_grad():
                        p_sel, c_sel = self._forward_gathered(net, net_device,
                                                              xin, flat)
                else:
                    with torch.no_grad():
                        p_sel, c_sel = self._forward_gathered(net, net_device,
                                                              xin, flat)
                # Respond per request, rows in request order: only the gathered
                # legal entries ever hit the (pickling) response queues.
                target_qs = self.resp_qs if net_id == 'live' else self.pool_resp_qs
                ri = 0
                for wid, o, ls in group:
                    out = []
                    for _ in ls:
                        a, b = offs[ri], offs[ri + 1]
                        out.append((p_sel[a:b], c_sel[a:b]))
                        ri += 1
                    target_qs[wid].put(out)

    def episodes(self):
        # Workers ship (samples, current lambda, n solver-labeled samples) —
        # the samples already include the solver-labeled auxiliary ones;
        # lambda / aux-count are surfaced for the training loop's logging.
        self.last_lam = 1.0
        self.last_aux = 0
        while True:
            samples, lam, n_aux = self.episode_q.get()
            self.last_lam = lam
            self.last_aux = n_aux
            yield samples

    def shutdown(self):
        self._stop.set()
        try:
            self.server.join(timeout=2.0)
        except Exception:
            pass
        for p in self.procs:
            p.terminate()
        for p in self.procs:
            p.join(timeout=2.0)
        for q in ([self.req_q, self.episode_q] + self.resp_qs
                  + self.pool_resp_qs):
            try:
                q.close(); q.cancel_join_thread()
            except Exception:
                pass

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# Ported from the Boop bayesian_mcts_3 notebook's proven defaults where the
# knob is algorithm-level (search/training mechanics); values that are clearly
# game-scale-dependent (network size, buffer, ply cap) are re-derived for
# chess below with the reasoning spelled out. Chess is vastly more complex
# than 6×6 Boop (game-tree complexity, 4674-action space, opening theory) —
# treat these as a reasonable STARTING point, not tuned defaults; expect to
# need far more self-play episodes/compute than the Boop notebooks to reach
# meaningful strength, and see the intro cell for chess-specific tuning ideas.
NUM_EPISODES     = 50_000  # run budget only — the LR schedule no longer
                           # depends on it, so extend freely
FULL_MCTS_SIMS   = 1000    # high-quality self-play (25% of games)
FAST_MCTS_SIMS   = 250     # fast self-play (75%) — more sims = sharper posteriors
FAST_GAME_PROB   = 0.75    # fraction of episodes using FAST_MCTS_SIMS
MAX_SELFPLAY_PLIES = 400   # self-play pacing safety valve only (chess is formally
                           # bounded by the 50-move rule but can still run very
                           # long, especially early in training) — SAFE to cut
                           # off: no training target here depends on the
                           # eventual game outcome z, only on that move's own
                           # root search belief, so an abandoned game's already-
                           # recorded samples stay fully valid.
# ── Hardware parallelism (AMD GPU via DirectML + 6-core CPU) ─────────────────
USE_WORKERS      = True    # multiprocessing self-play with a central GPU server
SELFPLAY_WORKERS = 4       # CPU game/tree processes
GAMES_PER_WORKER = 16      # per worker, split into two pipelined halves of 8
WORKER_WAVE      = 8       # leaves/game/wave; server batches all workers' requests
N_PARALLEL_GAMES = 8       # (single-process fallback when USE_WORKERS=False)
WAVE_PER_GAME    = 8       # MCTS leaves per game per wave
TEMP_THRESHOLD   = 20      # Thompson-sampled moves per game; posterior-mean
                           # argmax afterwards (the annealing analogue)
CONF_CAP         = 100.0   # posterior targets rescaled (mean-preserving) to at
                           # most this total count — caps certainty growth
                           # across generations (CONF_MAX clamps the net's side)
SELFPLAY_POOL_PROB = 0.15  # fraction of self-play games where one random side
                           # is played by a FROZEN benchmark instead of the live
                           # net mirror-matching itself — counters the
                           # non-transitive "strategy cycling" pure self-play can
                           # fall into (only the live side's moves are training
                           # targets). 0 disables it.
EVAL_EVERY       = 50      # QUICK eval: live net vs the most recent benchmark
DEEP_EVAL_EVERY  = 500     # DEEP eval: round-robin tournaments + new benchmark
QUICK_EVAL_GAMES = 20      # games vs the most recent benchmark (quick pulse)
TOURNEY_GAMES_PER_PAIR = 20  # round-robin games per pair (10 each colour)
TOURNEY_FULL_RR  = False   # linear-cost tournaments (new model vs everyone +
                           # REFRESH_PAIRS random old pairs)
REFRESH_PAIRS    = 4
# Independent evaluation tracks: (name, MCTS simulations per move). Each track
# keeps its OWN Elo table and W/D/L matrix. sims=0 is the raw network.
EVAL_TRACKS      = [('policy', 0), ('mcts25', 25), ('mcts100', 100)]
EVAL_OPENING_PLIES = 4     # random opening half-moves per eval game (variety)
START_ELO        = 1000.0  # random and the initial net seed here
K_FACTOR         = 32.0    # Elo update step
BATCH_SIZE       = 512
TRAIN_STEPS_PER_EP = 8     # gradient steps per self-play game
# No 8x symmetry augmentation for chess (see cell 5 / intro), and chess games
# run longer than Boop's (~50-70 plies vs ~30-40) — so at the SAME "how many
# recent episodes does the buffer remember" depth, the raw-position count is
# both undivided by 8 and multiplied by a longer game length. Sized here on
# the fresher side deliberately: a large, stale, unaugmented buffer was
# exactly what let the Boop bayesian_mcts_3 runs get stuck retraining on their
# own increasingly-confident-but-wrong past selves (see that notebook's
# training-log discussion) before a buffer flush (accidentally, via a
# checkpoint resume) broke the loop. ~100k positions / ~60 plies per game
# ≈ 1600 most-recent games of memory.
MAX_BUFFER       = 100_000
CHANNELS         = 128
NUM_BLOCKS       = 10      # deeper than Boop's 8: chess's 20x8x8 input and
                           # 4674-action space is far richer than Boop's 9x6x6/108
LR_PEAK          = 2e-3
LR_WARMUP_EPS    = 100      # linear warmup episodes
LR_DECAY_EPS     = 30_000   # cosine horizon — decoupled from NUM_EPISODES
WEIGHT_DECAY     = 1e-4
LR_MIN_FACTOR    = 0.10     # cosine decay floor (fraction of LR_PEAK)
CHECKPOINT_DIR   = 'chess_checkpoints_thompson'
RESUME_FROM_CHECKPOINT = True  # auto-resume from CHECKPOINT_DIR/latest.pt

# ── Setup ─────────────────────────────────────────────────────────────────────
game         = GAME                                           # from cell 2
base_network = ThompsonChessNet(CHANNELS, NUM_BLOCKS).to(DEVICE)  # uncompiled;
network      = base_network                                   # shares weights w/ compiled

# torch.compile: same policy as the Boop notebooks — skipped on DirectML and
# whenever Inductor has no C++ backend; falls back to eager on lazy failure.
network = base_network
if _BACKEND != 'directml' and hasattr(torch, 'compile'):
    import shutil, platform, logging
    _cc = (shutil.which('cl') if platform.system() == 'Windows'
           else shutil.which('g++') or shutil.which('gcc') or shutil.which('clang'))
    if _cc is None:
        print('torch.compile: skipped (no C++ compiler for Inductor) — eager mode')
    else:
        logging.getLogger('torch.fx.experimental.symbolic_shapes').setLevel(logging.ERROR)
        try:
            _compiled = torch.compile(base_network, dynamic=True)
            with torch.no_grad():                   # force compilation now
                _compiled(torch.zeros(1, *_OBS_SHAPE, device=DEVICE))
            network = _compiled
            print('torch.compile: enabled')
        except Exception as _e:
            print(f'torch.compile: disabled ({type(_e).__name__}) — running eager mode')

class LerpFreeAdamW(torch.optim.Optimizer):
    """AdamW built from mul_/add_/addcmul_/addcdiv_ only — DirectML lacks
    aten::lerp, which torch's AdamW uses in both its foreach and single-tensor
    paths (each step would round-trip every parameter through the CPU).
    Identical math to torch.optim.AdamW (decoupled decay, bias correction)."""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=1e-2):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps,
                                      weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            lr, (b1, b2) = group['lr'], group['betas']
            eps, wd = group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue
                st = self.state[p]
                if not st:
                    st['step'] = 0
                    st['exp_avg'] = torch.zeros_like(p)
                    st['exp_avg_sq'] = torch.zeros_like(p)
                st['step'] += 1
                t = int(st['step'])
                m, v = st['exp_avg'], st['exp_avg_sq']
                p.mul_(1.0 - lr * wd)                     # decoupled decay
                m.mul_(b1).add_(p.grad, alpha=1.0 - b1)   # lerp-free
                v.mul_(b2).addcmul_(p.grad, p.grad, value=1.0 - b2)
                bc1 = 1.0 - b1 ** t
                bc2 = 1.0 - b2 ** t
                denom = (v.sqrt() / (bc2 ** 0.5)).add_(eps)
                p.addcdiv_(m, denom, value=-(lr / bc1))
        return loss


if _BACKEND == 'directml':
    optimizer = LerpFreeAdamW(network.parameters(),
                              lr=LR_PEAK, weight_decay=WEIGHT_DECAY)
else:
    optimizer = torch.optim.AdamW(network.parameters(),
                                  lr=LR_PEAK, weight_decay=WEIGHT_DECAY)

def _lr_lambda(ep):
    if ep < LR_WARMUP_EPS:
        return ep / max(LR_WARMUP_EPS, 1)
    frac = min(1.0, (ep - LR_WARMUP_EPS) / max(LR_DECAY_EPS - LR_WARMUP_EPS, 1))
    # float(): np.cos returns a numpy scalar, which otherwise lands in the
    # scheduler state and trips torch.load(weights_only=True) on resume.
    return float(max(LR_MIN_FACTOR, 0.5 * (1.0 + np.cos(np.pi * frac))))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)


# ── Loss ──────────────────────────────────────────────────────────────────────
# Targets are SPARSE evidence counts (see root_target): per sample, the m
# evidence edges' action indices + their (win, draw, loss) pseudo-counts. The
# losses below therefore operate directly on GATHERED entries — p_sel (M, 3),
# c_sel (M,), t_sel (M, 3), where M = total evidence entries across the batch
# (~10-35 per position, i.e. ~0.5% of the dense B×4674 grid). This is exactly
# the same masked-mean the dense formulation computed (the mask selected these
# entries and zeroed the rest) but does ~200× less lgamma/digamma work — which
# on DirectML runs on the CPU (no DML kernel, see loss_backward) and was by far
# the most expensive part of a training step. Two losses are available
# (LOSS_FN) — the K=3 generalization of the Boop bayesian_mcts_3 notebook's
# Beta losses (verified numerically: KL(self)=0, the Dirichlet-Multinomial
# formula matches both its single-draw closed form and a 2M-sample Monte-Carlo
# estimate of the true marginal likelihood, and the gathered forms match the
# dense reference formulas exactly — see the test suite):
#
#   'kl'       KL( Dirichlet(evidence) ‖ Dirichlet(predicted prior) ). Matches
#              the whole distribution, including concentration — so it drives
#              predicted confidence toward the TARGET concentration whether or
#              not the predicted mean is right. That is the self-referential
#              path that lets confidence inflate without earning it.
#
#   'evidence' Negative Dirichlet-Multinomial marginal log-likelihood of the
#              evidence counts under the predicted prior. The prior is SCORED
#              against the evidence, never matched to a target containing
#              itself, so higher predicted confidence lowers the loss ONLY
#              where the predicted mean actually predicts the counts —
#              confidence has to be earned.

def dirichlet_kl_loss(p_sel, c_sel, t_sel):
    """KL( Dirichlet(target counts) ‖ Dirichlet(conf·probs) ), averaged over
    the gathered evidence entries. p_sel: (M,3); c_sel: (M,); t_sel: (M,3)."""
    t    = t_sel.clamp_min(ALPHA_FLOOR)
    b    = (c_sel.unsqueeze(-1) * p_sel).clamp_min(ALPHA_FLOOR)
    t0   = t.sum(dim=-1)
    b0   = b.sum(dim=-1)
    kl = (torch.lgamma(t0) - torch.lgamma(t).sum(dim=-1)
          - torch.lgamma(b0) + torch.lgamma(b).sum(dim=-1)
          + ((t - b) * (torch.digamma(t)
                        - torch.digamma(t0).unsqueeze(-1))).sum(dim=-1))
    return kl.mean()


def dirichlet_multinomial_loss(p_sel, c_sel, t_sel):
    """Negative Dirichlet-Multinomial marginal log-likelihood of the evidence
    counts `t_sel` (win, draw, loss) under the predicted prior
    Dirichlet(A), A = conf·probs:  −[logB(A+n) − logB(A)], with
    logB(x) = sum_k lgamma(x_k) − lgamma(sum_k x_k) the multivariate Beta
    function. Averaged over the gathered evidence entries.
    Non-self-referential: confidence is rewarded only when the predicted mean
    is right."""
    n = t_sel
    A = (c_sel.unsqueeze(-1) * p_sel).clamp_min(ALPHA_FLOOR)
    log_ev = ((torch.lgamma(A + n).sum(dim=-1) - torch.lgamma((A + n).sum(dim=-1)))
              - (torch.lgamma(A).sum(dim=-1) - torch.lgamma(A.sum(dim=-1))))
    return -log_ev.mean()


LOSS_FN = 'evidence'          # 'evidence' (Dirichlet-Multinomial) | 'kl' (for A/B tests)

def _search_loss(p_sel, c_sel, t_sel):
    return (dirichlet_multinomial_loss if LOSS_FN == 'evidence'
            else dirichlet_kl_loss)(p_sel, c_sel, t_sel)


# DirectML has no lgamma/digamma kernels (as with lerp / native_group_norm
# elsewhere in this notebook). Rather than gamble on a CPU fallback or on
# cross-device autograd, on that backend we SPLIT the backward at the device
# boundary: the gathered head outputs are copied to CPU, the loss and its
# gradient are computed there, and those gradients are handed back to the GPU
# graph as seed gradients. No lgamma runs on the GPU, and the only device
# transfers are plain tensor copies — never an autograd edge spanning devices.
# The gather itself runs on-device when index_select's backward passed the
# probe in the net cell (so only the M evidence entries — ~0.5% of the dense
# (B, 4674, ·) outputs — ever cross the bus); otherwise the dense outputs are
# downloaded once and gathered on CPU (the pre-optimization data volume, but
# still ~200× less lgamma work).
_KL_SPLIT = (_BACKEND == 'directml')

def loss_backward(probs, conf, flat_idx, t_sel):
    """Gathered search loss + backward into the network params. `probs`
    (B,A,3) / `conf` (B,A) are live on the training device with grad history;
    `flat_idx` (M,) int64 CPU indexes the flattened (B·A) action axis at the
    batch's evidence entries; `t_sel` (M,3) CPU holds their target counts.
    Returns (loss value, p_sel, c_sel) — detached CPU copies of the gathered
    predictions, reused by the training-loop diagnostics so nothing dense is
    ever downloaded. `optimizer.zero_grad()` must precede this;
    `clip_grad_norm_` + `optimizer.step()` follow it."""
    dev = probs.device
    if _KL_SPLIT:
        if _GATHER_BWD_OK:
            # On-device gather, CPU loss: seed gradients flow back through
            # index_select's backward (index_add) on the device.
            fi = flat_idx.to(dev)
            p_sel = probs.reshape(-1, 3).index_select(0, fi)
            c_sel = conf.reshape(-1).index_select(0, fi)
            p_cpu = p_sel.detach().cpu().requires_grad_(True)
            c_cpu = c_sel.detach().cpu().requires_grad_(True)
            loss_cpu = _search_loss(p_cpu, c_cpu, t_sel)
            loss_cpu.backward()
            torch.autograd.backward([p_sel, c_sel],
                                    [p_cpu.grad.to(dev), c_cpu.grad.to(dev)])
            return float(loss_cpu.item()), p_cpu.detach(), c_cpu.detach()
        # Fallback: download the dense head outputs once, gather on CPU, and
        # seed a dense (mostly-zero) gradient back through probs/conf.
        p_all = probs.detach().cpu().reshape(-1, 3)
        c_all = conf.detach().cpu().reshape(-1)
        p_cpu = p_all.index_select(0, flat_idx).requires_grad_(True)
        c_cpu = c_all.index_select(0, flat_idx).requires_grad_(True)
        loss_cpu = _search_loss(p_cpu, c_cpu, t_sel)
        loss_cpu.backward()
        gp = torch.zeros_like(p_all).index_add_(0, flat_idx, p_cpu.grad)
        gc = torch.zeros_like(c_all).index_add_(0, flat_idx, c_cpu.grad)
        torch.autograd.backward([probs, conf],
                                [gp.reshape(probs.shape).to(dev),
                                 gc.reshape(conf.shape).to(dev)])
        return float(loss_cpu.item()), p_cpu.detach(), c_cpu.detach()
    fi = flat_idx.to(dev)
    ts = t_sel.to(dev)
    p_sel = probs.reshape(-1, 3).index_select(0, fi)
    c_sel = conf.reshape(-1).index_select(0, fi)
    loss = _search_loss(p_sel, c_sel, ts)
    loss.backward()
    return float(loss.item()), p_sel.detach().cpu(), c_sel.detach().cpu()


# Self-play source: worker pool (multiprocess + GPU server) or in-process.
import os
import threading
try:
    self_play.shutdown()               # re-running this cell: stop old workers
except (NameError, AttributeError):
    pass
if USE_WORKERS:
    _wcfg = dict(seed=int(np.random.randint(1_000_000)),
                 games_per_worker=GAMES_PER_WORKER, wave=WORKER_WAVE,
                 fast_sims=FAST_MCTS_SIMS, full_sims=FULL_MCTS_SIMS,
                 fast_prob=FAST_GAME_PROB, temp_threshold=TEMP_THRESHOLD,
                 conf_cap=CONF_CAP, pool_prob=SELFPLAY_POOL_PROB,
                 checkpoint_dir=CHECKPOINT_DIR,
                 max_selfplay_plies=MAX_SELFPLAY_PLIES)
    self_play = MPSelfPlayPool(network, DEVICE, SELFPLAY_WORKERS, _wcfg,
                               checkpoint_dir=CHECKPOINT_DIR,
                               channels=CHANNELS, num_blocks=NUM_BLOCKS)
    torch.set_num_threads(max(2, (os.cpu_count() or 6) - SELFPLAY_WORKERS))
else:
    self_play = ThompsonParallelSelfPlay(game, network, DEVICE,
                                         n_parallel=N_PARALLEL_GAMES,
                                         wave_per_game=WAVE_PER_GAME,
                                         fast_sims=FAST_MCTS_SIMS,
                                         full_sims=FULL_MCTS_SIMS,
                                         fast_prob=FAST_GAME_PROB,
                                         temp_threshold=TEMP_THRESHOLD,
                                         conf_cap=CONF_CAP,
                                         pool_prob=SELFPLAY_POOL_PROB,
                                         checkpoint_dir=CHECKPOINT_DIR,
                                         channels=CHANNELS,
                                         num_blocks=NUM_BLOCKS,
                                         max_selfplay_plies=MAX_SELFPLAY_PLIES)
episode_stream = self_play.episodes()
# Serializes model access between the inference-server thread and training.
MODEL_LOCK = getattr(self_play, 'lock', None) or threading.Lock()

# Small-batch inference (eval games, tournaments) runs on CPU replicas — batch-1
# forwards on DirectML are latency-bound and would dominate eval time.
EVAL_DEVICE = 'cpu'

def _cpu_sd(net):
    return {k: v.detach().cpu() for k, v in net.state_dict().items()}

def _cpu_clone(net):
    m = ThompsonChessNet(CHANNELS, NUM_BLOCKS)
    m.load_state_dict(_cpu_sd(net))
    m.eval()
    return m

def _to_cpu_tree(obj):
    if torch.is_tensor(obj):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: _to_cpu_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_cpu_tree(v) for v in obj]
    return obj

replay_buffer = []
# Tournament pool. `random` and the untrained `0` net both start at START_ELO.
_init_snap = _cpu_clone(base_network)
eval_net   = _cpu_clone(base_network)  # refreshed at every eval
benchmarks = [{'label': 'random', 'net': None},
              {'label': '0',      'net': _init_snap}]
# Per-track Elo tables and head-to-head matrices (independent between tracks).
elos = {name: {'random': START_ELO, '0': START_ELO} for name, _ in EVAL_TRACKS}
wdl  = {name: defaultdict(lambda: [0, 0, 0])         for name, _ in EVAL_TRACKS}
hist = {'ep': [], 'loss': [], 'v_err': [], 'conf_t': [], 'conf_p': [],
        'lam': [],                       # innovation-calibration factor (cell 5)
        'quick_ep': [], 'q_w': [], 'q_d': [], 'q_l': [],
        'deep_ep': [], 'elo_snap': []}   # elo_snap: list of {track: {label: elo}}

# ── Checkpointing: survive crashes, resume indefinitely ──────────────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
_LATEST_CKPT = os.path.join(CHECKPOINT_DIR, 'latest.pt')

def _save_benchmark_net(label, net):
    if net is not None:
        torch.save(net.state_dict(),
                   os.path.join(CHECKPOINT_DIR, f'bench_{label}.pt'))

def save_checkpoint(ep):
    # Copy state to CPU first: DirectML (privateuseone) tensors may not pickle.
    with MODEL_LOCK:
        _model_sd = _to_cpu_tree(base_network.state_dict())
        _opt_sd   = _to_cpu_tree(optimizer.state_dict())
    torch.save({'ep': ep,
                'model': _model_sd,
                'optimizer': _opt_sd,
                'scheduler': scheduler.state_dict(),
                'elos': elos,
                'wdl': {name: dict(w) for name, w in wdl.items()},
                'hist': hist,
                'bench_labels': [b['label'] for b in benchmarks]}, _LATEST_CKPT)

start_ep = 1
if RESUME_FROM_CHECKPOINT and os.path.exists(_LATEST_CKPT):
    try:
        _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=True)
    except Exception:                     # older ckpt with a numpy scalar inside
        _ck = torch.load(_LATEST_CKPT, map_location='cpu', weights_only=False)
    base_network.load_state_dict(_ck['model'])
    optimizer.load_state_dict(_ck['optimizer'])
    scheduler.load_state_dict(_ck['scheduler'])
    elos = _ck['elos']
    wdl  = {name: defaultdict(lambda: [0, 0, 0], w)
            for name, w in _ck['wdl'].items()}
    hist = _ck['hist']
    hist.setdefault('lam', [float('nan')] * len(hist['ep']))
    benchmarks = [{'label': 'random', 'net': None}]
    for _lbl in _ck['bench_labels']:
        if _lbl == 'random':
            continue
        _bn = ThompsonChessNet(CHANNELS, NUM_BLOCKS)  # benchmarks live on CPU
        _bn.load_state_dict(torch.load(
            os.path.join(CHECKPOINT_DIR, f'bench_{_lbl}.pt'),
            map_location='cpu', weights_only=True))
        _bn.eval()
        benchmarks.append({'label': _lbl, 'net': _bn})
    start_ep = _ck['ep'] + 1
    print(f'Resumed from checkpoint: episode {_ck["ep"]}, pool {len(benchmarks)}')


def _run_all_tracks(focus=None):
    for name, sims in EVAL_TRACKS:
        run_tournament(benchmarks, elos[name], wdl[name], game, EVAL_DEVICE,
                       TOURNEY_GAMES_PER_PAIR, K_FACTOR, sims,
                       opening_plies=EVAL_OPENING_PLIES,
                       focus_label=focus, refresh_pairs=REFRESH_PAIRS)
    hist['deep_ep'].append(ep_now[0])
    hist['elo_snap'].append({name: dict(elos[name]) for name, _ in EVAL_TRACKS})


def _print_tracks(prefix):
    for name, _ in EVAL_TRACKS:
        ladder = '  '.join(f'{b["label"]}={elos[name][b["label"]]:.0f}'
                           for b in sorted(benchmarks, key=lambda b: -elos[name][b['label']]))
        print(f'{prefix} {name:>7} Elos (strong→weak): {ladder}')


n_params = sum(p.numel() for p in network.parameters() if p.requires_grad)
print(f'Device: {DEVICE}  |  ThompsonChessNet params: {n_params:,}')
print(f'{NUM_EPISODES} eps | fast {FAST_MCTS_SIMS} sims ({FAST_GAME_PROB:.0%}) '
      f'/ full {FULL_MCTS_SIMS} sims | mixture-prop + MCTS-Solver | {LOSS_FN} loss, '
      f'conf cap {CONF_CAP:.0f} | pool {SELFPLAY_POOL_PROB:.0%} | no aug | '
      f'{TRAIN_STEPS_PER_EP} steps/ep | ply cap {MAX_SELFPLAY_PLIES}')
print(f'Quick eval every {EVAL_EVERY} eps (vs latest benchmark); deep tournaments '
      f'every {DEEP_EVAL_EVERY} eps | tracks: {[f"{n}({s})" for n, s in EVAL_TRACKS]}')
print(f'Checkpoints: {CHECKPOINT_DIR} (resume={RESUME_FROM_CHECKPOINT})')
if USE_WORKERS:
    print(f'Self-play: {SELFPLAY_WORKERS} worker processes × {GAMES_PER_WORKER} games '
          f'(wave {WORKER_WAVE}) → GPU inference server on {DEVICE}')
print('-' * 72)

aux_total = 0        # solver-labeled auxiliary samples seen (see cell 5)
ep_now = [start_ep - 1]
if start_ep == 1:
    # Iteration-0 tournaments: random vs the untrained net, one per track.
    _save_benchmark_net('0', _init_snap)
    _run_all_tracks()
    _print_tracks('Iter     0 |')
    save_checkpoint(0)
print('-' * 72)

for ep in range(start_ep, NUM_EPISODES + 1):
    ep_now[0] = ep

    # 1. Pull the next finished self-play game. Its samples are the per-move
    #    root targets PLUS any solver-labeled auxiliary samples (positions the
    #    MCTS-Solver proved during the game's searches — exact game-theoretic
    #    labels, same (obs, idx, counts) format; see cell 5).
    network.eval()
    raw_data = next(episode_stream)
    aux_total += int(getattr(self_play, 'last_aux', 0))

    # 2. No board-symmetry augmentation for chess (see cell 5) — augment_sample
    #    is a no-op wrapper kept only so this call-site matches the Boop
    #    notebooks structurally. Samples are (obs fp16, action_idx, counts):
    #    sparse targets + half-precision observations keep the buffer at
    #    ~3 KB/sample (~300 MB at MAX_BUFFER=100k) instead of the ~61 KB/sample
    #    (~6 GB!) a dense (4674, 3) fp32 target per move would cost.
    for obs, ti, tc in raw_data:
        replay_buffer.extend(augment_sample(obs, ti, tc))
    if len(replay_buffer) > MAX_BUFFER:
        del replay_buffer[:-MAX_BUFFER]

    # 3. Train: the selected search loss (see LOSS_FN) on evidence-only targets
    network.train()
    losses, v_errs, conf_ts, conf_ps = [], [], [], []
    if len(replay_buffer) >= BATCH_SIZE:
        for _ in range(TRAIN_STEPS_PER_EP):
            batch = random.sample(replay_buffer, BATCH_SIZE)
            # Assemble the batch's evidence entries: flat indices into the
            # (B·A) action axis + their target counts, all on CPU (tiny).
            fi_parts, t_parts = [], []
            for b, (_, ti, tc) in enumerate(batch):
                if len(ti):
                    fi_parts.append(ti.astype(np.int64) + b * _NUM_ACTIONS)
                    t_parts.append(tc)
            if not fi_parts:
                continue                       # no evidence in batch (unlikely)
            flat_idx = torch.from_numpy(np.concatenate(fi_parts))
            t_sel    = torch.from_numpy(
                           np.concatenate(t_parts).astype(np.float32))
            # The device part of the step sits under the lock: every device op
            # must be serialized against the inference-server thread (DirectML
            # is not thread-safe).
            with MODEL_LOCK:
                x_b = batch_to_tensor([s[0] for s in batch], DEVICE)
                probs, conf = network(x_b)
                optimizer.zero_grad()
                loss_val, p_sel, c_sel = loss_backward(probs, conf,
                                                       flat_idx, t_sel)
                torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                optimizer.step()

            with torch.no_grad():
                # Diagnostics on the gathered CPU copies loss_backward already
                # produced — nothing dense is downloaded, no exotic op ever
                # touches the DirectML device, and this runs OUTSIDE the lock.
                t  = t_sel.clamp_min(ALPHA_FLOOR)
                t0 = t.sum(dim=-1)
                tv = (t[:, 0] - t[:, 2]) / t0              # target mean value
                pv = p_sel[:, 0] - p_sel[:, 2]             # predicted mean value
                v_errs.append(float((tv - pv).abs().mean()))
                conf_ts.append(float(t0.mean()))
                conf_ps.append(float(c_sel.mean()))
            losses.append(loss_val)
        scheduler.step()

    if ep % EVAL_EVERY != 0:
        continue

    # 4. Evaluation (CPU replica: batch-1 DirectML inference is latency-bound)
    network.eval()
    with MODEL_LOCK:
        eval_net.load_state_dict(_cpu_sd(base_network))
    mloss = float(np.mean(losses))  if losses  else float('nan')
    mve   = float(np.mean(v_errs))  if v_errs  else float('nan')
    mct   = float(np.mean(conf_ts)) if conf_ts else float('nan')
    mcp   = float(np.mean(conf_ps)) if conf_ps else float('nan')
    lam_now = float(getattr(self_play, 'last_lam', float('nan')))
    hist['ep'].append(ep); hist['loss'].append(mloss); hist['v_err'].append(mve)
    hist['conf_t'].append(mct); hist['conf_p'].append(mcp)
    hist['lam'].append(lam_now)
    lr_now = optimizer.param_groups[0]['lr']

    if ep % DEEP_EVAL_EVERY == 0:
        # DEEP: add current net as a new benchmark, warm-starting each track's
        # Elo from the previous checkpoint, then run one round-robin per track.
        new_label = str(ep)
        for name, _ in EVAL_TRACKS:
            elos[name][new_label] = elos[name][benchmarks[-1]['label']]
        with MODEL_LOCK:
            snap = _cpu_clone(base_network)
        benchmarks.append({'label': new_label, 'net': snap})
        _run_all_tracks(None if TOURNEY_FULL_RR else new_label)
        print(f'Ep {ep:5d} | loss={mloss:.3f} vErr={mve:.3f} a0 t/p={mct:.1f}/{mcp:.1f} '
              f'lam={lam_now:.2f} aux={aux_total} '
              f'| DEEP tournaments (pool {len(benchmarks)}) '
              f'| lr={lr_now:.2e}')
        _print_tracks('        |')
        _save_benchmark_net(new_label, snap)
        save_checkpoint(ep)
    else:
        # QUICK: search-free pulse vs the most recent benchmark (no Elo change).
        ref = benchmarks[-1]
        w, d, l = play_match(eval_net, ref['net'], game,
                             QUICK_EVAL_GAMES, EVAL_DEVICE)
        hist['quick_ep'].append(ep)
        hist['q_w'].append(w); hist['q_d'].append(d); hist['q_l'].append(l)
        print(f'Ep {ep:5d} | loss={mloss:.3f} vErr={mve:.3f} a0 t/p={mct:.1f}/{mcp:.1f} '
              f'lam={lam_now:.2f} aux={aux_total} '
              f'| vs {ref["label"]:>5} W{w:2d}D{d:2d}L{l:2d} '
              f'(policy Elo={elos["policy"][ref["label"]]:.0f}) | lr={lr_now:.2e}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle('Chess ThompsonZero Training', fontsize=14)

ax = axes[0, 0]
ax.plot(hist['ep'], hist['loss'], label=f'{LOSS_FN} loss')
ax.set_title('Search loss (falls for KL; Dirichlet-Multinomial NLL is a different scale)')
ax.set_xlabel('Episode')
ax2 = ax.twinx()
ax2.plot(hist['ep'], hist['conf_t'], color='tab:purple', alpha=0.6, label='α₀ target')
ax2.plot(hist['ep'], hist['conf_p'], color='tab:brown', alpha=0.6, label='α₀ predicted')
ax2.set_ylabel('mean α₀')
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8)

axes[0, 1].plot(hist['ep'], hist['v_err'], color='tab:orange', label='|v err|')
axes[0, 1].set_title('|value target − predicted|  +  calibration λ')
axes[0, 1].set_xlabel('Episode')
if any(np.isfinite(hist.get('lam', []))):
    axl = axes[0, 1].twinx()
    axl.plot(hist['ep'], hist['lam'], color='tab:blue', alpha=0.6, label='λ (calib)')
    axl.axhline(1.0, color='tab:blue', linestyle=':', linewidth=0.8, alpha=0.5)
    axl.set_ylabel('λ  (1 = calibrated)')
    h1, l1 = axes[0, 1].get_legend_handles_labels()
    h2, l2 = axl.get_legend_handles_labels()
    axes[0, 1].legend(h1 + h2, l1 + l2, fontsize=8)

# Elo of the model trained-to-episode-N, one line per evaluation track.
def _newest_label(ep):
    return '0' if ep == 0 else str(ep)
for name, _ in EVAL_TRACKS:
    xs, ys = [], []
    for ep, snap in zip(hist['deep_ep'], hist['elo_snap']):
        lbl = _newest_label(ep)
        if lbl in snap[name]:
            xs.append(ep); ys.append(snap[name][lbl])
    axes[0, 2].plot(xs, ys, marker='.', label=name)
axes[0, 2].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[0, 2].set_title('Current-model Elo by track')
axes[0, 2].set_xlabel('Episode'); axes[0, 2].legend(fontsize=8)

# Quick-eval progress pulse: search-free W/D/L vs the most recent benchmark.
axes[1, 0].plot(hist['quick_ep'], hist['q_w'], color='tab:green', marker='.', label='Win')
axes[1, 0].plot(hist['quick_ep'], hist['q_d'], color='gray', linestyle='--', label='Draw')
axes[1, 0].plot(hist['quick_ep'], hist['q_l'], color='tab:red', linestyle=':', label='Loss')
axes[1, 0].set_title(f'Quick eval vs latest benchmark ({QUICK_EVAL_GAMES} games, search-free)')
axes[1, 0].set_xlabel('Episode'); axes[1, 0].legend(fontsize=8)

# Final Elo per benchmark, grouped bars by track.
labels = [b['label'] for b in benchmarks]
x = np.arange(len(labels)); width = 0.8 / max(len(EVAL_TRACKS), 1)
for k, (name, _) in enumerate(EVAL_TRACKS):
    vals = [elos[name].get(l, np.nan) for l in labels]
    axes[1, 1].bar(x + (k - (len(EVAL_TRACKS) - 1) / 2) * width, vals, width, label=name)
axes[1, 1].axhline(START_ELO, color='gray', linestyle='--', linewidth=0.8)
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 1].set_title('Final Elo per benchmark'); axes[1, 1].set_ylabel('Elo')
axes[1, 1].legend(fontsize=8)

# Head-to-head win-rate matrix for the strongest track (highest sims).
mtrack = EVAL_TRACKS[-1][0]
W = wdl[mtrack]
n = len(labels)
M = np.full((n, n), np.nan)
for i, ri in enumerate(labels):
    for j, cj in enumerate(labels):
        if (ri, cj) in W:
            w, d, l = W[(ri, cj)]; tot = w + d + l
            if tot: M[i, j] = (w + 0.5 * d) / tot
        elif (cj, ri) in W:
            w, d, l = W[(cj, ri)]; tot = w + d + l
            if tot: M[i, j] = 1.0 - (w + 0.5 * d) / tot
im = axes[1, 2].imshow(M, cmap='RdYlGn', vmin=0, vmax=1)
axes[1, 2].set_xticks(range(n)); axes[1, 2].set_yticks(range(n))
axes[1, 2].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1, 2].set_yticklabels(labels, fontsize=7)
axes[1, 2].set_title(f'Win-rate matrix — {mtrack} (row vs col)')
for i in range(n):
    for j in range(n):
        if not np.isnan(M[i, j]):
            axes[1, 2].text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                            fontsize=6, color='black')
fig.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Arena: pit any two checkpoints against each other

There's no existing "KataZero-chess" notebook to compare against the way the
Boop notebooks compare ThompsonZero to KataZero, so this section is a general
**checkpoint-vs-checkpoint arena** instead: load any two `ThompsonChessNet`
state dicts (any `bench_<ep>.pt` from `CHECKPOINT_DIR`, or `latest.pt`'s
`['model']` entry) and play them off, search-free or with search, alternating
colours, with a printable move-by-move game log (chess positions are
human-inspectable via FEN/SAN, unlike Boop's board).

The training cell's own DEEP-eval tournaments already give you an ongoing Elo
ladder across generations (that *is* the primary strength signal — see the
plots cell); this section is for **ad-hoc** comparisons: two specific
checkpoints, a specific `LOSS_FN` variant against another, or a specific sim
budget.

**Plugging in an external opponent** (Stockfish via UCI, another engine, a
human): `arena()` below is specifically for two `ThompsonChessNet` checkpoints.
To face something else, write your own move loop around a `pick(state) ->
action` function for the external side (any `state.legal_actions()`-compatible
choice) — `arena()`'s body is short and is a template for that; nothing here
has been tested against an external engine, so treat this as a starting point,
not a ready integration.

In [ ]:
import os


def _load_sd(path):
    # Full training checkpoints (latest.pt) also store non-tensor objects — e.g.
    # a numpy float in the LR-scheduler state (np.cos in the LR schedule) — which
    # torch's default weights_only=True unpickler rejects. Try the safe load,
    # then fall back to a full load for these trusted, locally-produced files.
    try:
        sd = torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        sd = torch.load(path, map_location='cpu', weights_only=False)
    return sd['model'] if isinstance(sd, dict) and 'model' in sd else sd


def load_chess_net(path, channels=CHANNELS, num_blocks=NUM_BLOCKS):
    net = ThompsonChessNet(channels, num_blocks)
    net.load_state_dict(_load_sd(path))
    net.eval()
    return net


def arena(net_a, net_b, game, n_games=20, a_sims=0, b_sims=0,
          device='cpu', opening_plies=4, verbose=True):
    """net_a vs net_b. sims == 0 -> search-free (prior-mean argmax); sims > 0
    -> full Thompson-MCTS search (posterior-mean argmax, no root noise needed).
    Alternates colours; the first `opening_plies` moves of each game are random
    for variety. Returns (wins_a, draws, wins_b) from net_a's perspective."""
    bot_a = (make_thompson_bot(game, net_a, device, a_sims,
                               batch_size=max(1, min(a_sims, 16)))
             if a_sims > 0 else None)
    bot_b = (make_thompson_bot(game, net_b, device, b_sims,
                               batch_size=max(1, min(b_sims, 16)))
             if b_sims > 0 else None)
    w = d = l = 0
    for i in range(n_games):
        state = game.new_initial_state()
        a_player = i % 2                 # alternate colours
        ply = 0
        while not state.is_terminal() and ply < MAX_EVAL_PLIES:
            if ply < opening_plies:
                action = random.choice(state.legal_actions())
            elif state.current_player() == a_player:
                action = (_mcts_move(bot_a, state) if bot_a is not None
                          else policy_action(net_a, state, device))
            else:
                action = (_mcts_move(bot_b, state) if bot_b is not None
                          else policy_action(net_b, state, device))
            state.apply_action(action)
            ply += 1
        # A non-terminal state's returns() is exactly [0.0, 0.0] in chess, so a
        # ply-cap cutoff scores as a draw here with no special-casing needed.
        r = state.returns()[a_player]
        if r > 0:   w += 1
        elif r < 0: l += 1
        else:       d += 1
        if verbose and (i + 1) % 5 == 0:
            print(f'    {i + 1:3d}/{n_games}  net_a W{w} D{d} L{l}')
    return w, d, l


def play_and_show(net_a, net_b, game, device='cpu', max_plies=200):
    """One game, net_a (White) vs net_b (Black), search-free, printing each
    move in SAN and the final result — a quick sanity check you can eyeball.
    Either net may be None for a random mover (e.g. no second checkpoint yet)."""
    state = game.new_initial_state()
    moves = []
    ply = 0
    while not state.is_terminal() and ply < max_plies:
        mover = state.current_player()
        net = net_a if mover == 1 else net_b     # player 1 = White (see cell 2)
        action = (random.choice(state.legal_actions()) if net is None
                  else policy_action(net, state, device))
        moves.append(state.action_to_string(mover, action))
        state.apply_action(action)
        ply += 1
    print(' '.join(f'{i//2+1}.{m}' if i % 2 == 0 else m
                   for i, m in enumerate(moves)))
    if state.is_terminal():
        print('Result:', state.returns(), '  FEN:', state)
    else:
        print(f'(stopped at the {max_plies}-ply display cap, not terminal)')
    return state


# ── Run it ────────────────────────────────────────────────────────────────────
# By default: the LATEST checkpoint vs the UNTRAINED episode-0 net and vs
# random, using whatever this session has already trained/loaded. Point
# CKPT_A / CKPT_B at any two bench_<ep>.pt files (or another CHECKPOINT_DIR
# entirely) to compare specific generations or LOSS_FN variants.
CKPT_A       = os.path.join(CHECKPOINT_DIR, 'latest.pt')   # newest
CKPT_B       = os.path.join(CHECKPOINT_DIR, 'bench_0.pt')  # untrained baseline
ARENA_GAMES  = 20
ARENA_SIMS   = 50     # per-move simulations for the with-search match
ARENA_DEVICE = 'cpu'  # batch-1 inference: CPU beats DirectML here

if not os.path.exists(CKPT_A):
    print(f'No checkpoint at {CKPT_A} yet — train at least one episode first.')
else:
    net_a = load_chess_net(CKPT_A)
    net_b = load_chess_net(CKPT_B) if os.path.exists(CKPT_B) else None
    print(f'net_a: {CKPT_A}')
    print(f'net_b: {CKPT_B if net_b is not None else "(missing — using random)"}')

    if net_b is not None:
        print(f'\nSearch-free ({ARENA_GAMES} games):')
        w, d, l = arena(net_a, net_b, game, ARENA_GAMES, 0, 0, ARENA_DEVICE)
        print(f'  net_a W{w} D{d} L{l}  '
              f'({(w + 0.5 * d) / max(w + d + l, 1):.1%} score)')

        print(f'\n{ARENA_SIMS}-sim MCTS ({ARENA_GAMES} games):')
        w, d, l = arena(net_a, net_b, game, ARENA_GAMES,
                        ARENA_SIMS, ARENA_SIMS, ARENA_DEVICE)
        print(f'  net_a W{w} D{d} L{l}  '
              f'({(w + 0.5 * d) / max(w + d + l, 1):.1%} score)')

    print('\nOne search-free game vs random, shown move-by-move:')
    play_and_show(net_a, None if net_b is None else net_b, game, ARENA_DEVICE)